# Notebook 02 — Model Development, Validation, Calibration, and Robustness (Reviewer-Response Version)

This notebook retains the original baseline comparison and adds the reviewer-requested analyses:

- complete classical-model and MLP configuration documentation;
- baseline missingness audit;
- XGBoost/MLP paired uncertainty comparison;
- learning-curve analysis across increasing synthetic dataset sizes;
- prevalence sensitivity at 20%, 30%, 40%, and 50% delayed;
- MLP class-weight sensitivity;
- rule ablation and an alternative structural data-generating mechanism;
- probability calibration (Brier score and calibration curves), including distribution-shift evaluation;
- reproducible environment and hardware metadata.

All robustness conclusions remain **internal to the synthetic benchmark** and do not establish external validity.


In [ ]:
# =========================
# 0. Setup and Reproducibility
# =========================
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import json
import random
import platform
from time import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.inspection import permutation_importance
import sklearn
import joblib

# Baseline seed retained from the original experiment.
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Extended validation settings.
CV_SPLITS = 5
VALIDATION_SEEDS = [42, 52, 62]
RUN_EXTENDED_VALIDATION = True
RUN_LEARNING_CURVE = True
RUN_PREVALENCE_SENSITIVITY = True
RUN_STRUCTURAL_ROBUSTNESS = True
RUN_CALIBRATION_ANALYSIS = True
RUN_MLP_WEIGHT_SENSITIVITY = True
RUN_PAIRED_BOOTSTRAP = True

LEARNING_CURVE_TOTAL_SIZES = [5_000, 10_000, 50_000, 100_000, 500_000]
PREVALENCE_LEVELS = [0.20, 0.30, 0.40, 0.50]
BOOTSTRAP_REPS = 1000

# Repository-root resolver: allows notebooks to run from the repository root
# or from the notebooks/ directory without changing output paths.
def _find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".project-root").exists():
            return candidate
    return current

REPO_ROOT = _find_repo_root()
PROJECT_DIR = REPO_ROOT / "project_delay_outputs"
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
PLOTS_DIR = RESULTS_DIR / "plots"

for d in [DATA_DIR, MODEL_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

software_versions = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "baseline_seed": SEED,
    "cv_splits": CV_SPLITS,
    "validation_seeds": VALIDATION_SEEDS,
}

print("Setup complete")
print("Output directory:", PROJECT_DIR.resolve())
print("Validation settings:", {k: software_versions[k] for k in ["baseline_seed", "cv_splits", "validation_seeds"]})


In [ ]:
# =========================
# 1. Load Dataset
# =========================
data_path = DATA_DIR / "synthetic_project_delay_dataset.csv"

if not data_path.exists():
    raise FileNotFoundError(
        "Dataset not found. Run Notebook 01 first to create "
        "project_delay_outputs/data/synthetic_project_delay_dataset.csv"
    )

df = pd.read_csv(data_path)
print("Dataset shape:", df.shape)
display(df.head())
display(df.info())


In [ ]:
# =========================
# 1.1 Baseline Missingness Audit
# =========================
missing_by_column = df.isna().sum().sort_values(ascending=False)
baseline_missing_values_total = int(missing_by_column.sum())

print("Total missing values in generated baseline dataset:", baseline_missing_values_total)
if baseline_missing_values_total == 0:
    print(
        "The synthetic baseline contains no missing values. "
        "Imputation is retained only to keep the pipeline reusable for future real datasets."
    )
else:
    display(missing_by_column[missing_by_column > 0])

pd.DataFrame({
    "Column": missing_by_column.index,
    "Missing_Count": missing_by_column.values
}).to_csv(
    RESULTS_DIR / "FINAL_baseline_missingness_audit.csv",
    index=False
)


In [ ]:
# =========================
# 2. Select Features and Target
# =========================
target = "Delayed"

# Exclude leakage or post-outcome fields.
# These fields should not be used as model inputs because they are known after project execution
# or directly encode the target label.
excluded_columns = [
    "Project_ID",
    "Delayed",
    "Delay_Risk_Level",
    "Delay_Probability_True",
    "Actual_Duration_Days",
    "Time_Overrun_Percent",
    "Rule_Score"
]

missing_target = target not in df.columns
if missing_target:
    raise KeyError(f"Target column '{target}' was not found in the dataset.")

features = [c for c in df.columns if c not in excluded_columns]
X = df[features].copy()
y = df[target].astype(int)

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Target:", target)
print("Number of features:", len(features))
print("Categorical features:", categorical_features)
print("Numeric features:", len(numeric_features))
print("Class distribution:")
display(y.value_counts().rename({0: "Not Delayed", 1: "Delayed"}))
print("Class percentage:")
display((y.value_counts(normalize=True) * 100).round(3).rename({0: "Not Delayed", 1: "Delayed"}))


In [ ]:
# =========================
# 3. Preprocessing Pipeline and Baseline 80/20 Split
# =========================
# The 80/20 split is retained as the common baseline comparison used for every model.
# Extended validation is performed later for the selected best model.
try:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot_encoder)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("Baseline training rows:", X_train.shape[0])
print("Baseline testing rows:", X_test.shape[0])
print("Baseline split seed:", SEED)


In [ ]:
# =========================
# 4. Utility Functions
# =========================
def get_positive_probability(model_object, X_input):
    """Return probability for the positive class Delayed=1."""
    if hasattr(model_object, "predict_proba"):
        return model_object.predict_proba(X_input)[:, 1]
    if hasattr(model_object, "decision_function"):
        scores = model_object.decision_function(X_input)
        return 1 / (1 + np.exp(-scores))
    return model_object.predict(X_input)


def evaluate_model(model_name, y_true, y_pred, y_proba):
    """Create one comparable metrics row."""
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1_Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_proba),
    }


def save_classification_report(model_name, y_true, y_pred):
    """Print and save classification report."""
    print("=" * 80)
    print(f"Classification Report: {model_name}")
    print("=" * 80)
    print(classification_report(y_true, y_pred, target_names=["Not Delayed", "Delayed"], digits=6))

    report_dict = classification_report(
        y_true, y_pred,
        target_names=["Not Delayed", "Delayed"],
        digits=6,
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).T
    safe_name = model_name.lower().replace(" ", "_").replace("/", "_")
    report_df.to_csv(RESULTS_DIR / f"classification_report_{safe_name}.csv")
    return report_df


def plot_confusion_matrix(model_name, y_true, y_pred):
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        display_labels=["Not Delayed", "Delayed"],
        cmap="Blues",
        values_format="d",
        ax=ax
    )
    ax.set_title(f"Confusion Matrix - {model_name}")
    plt.tight_layout()
    safe_name = model_name.lower().replace(" ", "_").replace("/", "_")
    plt.savefig(PLOTS_DIR / f"confusion_matrix_{safe_name}.png", dpi=300, bbox_inches="tight")
    plt.show()


def plot_single_roc(model_name, y_true, y_proba):
    fig, ax = plt.subplots(figsize=(6, 5))
    RocCurveDisplay.from_predictions(y_true, y_proba, ax=ax)
    ax.set_title(f"ROC Curve - {model_name}")
    plt.tight_layout()
    safe_name = model_name.lower().replace(" ", "_").replace("/", "_")
    plt.savefig(PLOTS_DIR / f"roc_curve_{safe_name}.png", dpi=300, bbox_inches="tight")
    plt.show()


def get_transformed_feature_names(fitted_preprocessor):
    """Return feature names after preprocessing."""
    try:
        return fitted_preprocessor.get_feature_names_out().tolist()
    except Exception:
        names = []
        names.extend(numeric_features)
        if len(categorical_features) > 0:
            cat_pipe = fitted_preprocessor.named_transformers_["cat"]
            onehot = cat_pipe.named_steps["onehot"]
            names.extend(onehot.get_feature_names_out(categorical_features).tolist())
        return names


def to_dense(matrix):
    """Convert sparse matrix to dense array when needed."""
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)


def safe_sample_frame(X_frame, n=500):
    n = min(n, len(X_frame))
    return X_frame.sample(n=n, random_state=SEED)


In [ ]:
# =========================
# 5. Define Classical ML Models
# =========================
# Hyperparameters are kept consistent with the original baseline experiment.
ml_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=SEED
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=SEED
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=SEED
    ),
}

# Optional XGBoost if installed.
try:
    import xgboost
    from xgboost import XGBClassifier

    ml_models["XGBoost"] = XGBClassifier(
        n_estimators=350,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.90,
        colsample_bytree=0.90,
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1
    )
    software_versions["xgboost"] = xgboost.__version__
    print("XGBoost available and added:", xgboost.__version__)
except Exception as e:
    print("XGBoost not available. Continuing without it.")
    print("Reason:", e)

print("Models to train:")
for m in ml_models:
    print("-", m)


## 5.1 Model Hyperparameter Documentation

The complete estimator parameter dictionaries are exported directly from the fitted estimator definitions rather than transcribed manually. This prevents discrepancies between the code and the manuscript.

For the deep-learning model, the architecture and training settings are defined explicitly in `MLP_CONFIG` and exported as JSON.


In [ ]:
# =========================
# 5.1 Export Classical Model Settings
# =========================
classical_model_parameters = {
    name: model.get_params(deep=True)
    for name, model in ml_models.items()
}

with open(RESULTS_DIR / "classical_model_hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(classical_model_parameters, f, indent=2, default=str)

# Compact table containing the parameters most useful for the manuscript.
parameter_rows = []
for name, params in classical_model_parameters.items():
    for parameter, value in params.items():
        parameter_rows.append({
            "Model": name,
            "Parameter": parameter,
            "Value": value
        })

classical_parameter_df = pd.DataFrame(parameter_rows)
classical_parameter_df.to_csv(
    RESULTS_DIR / "classical_model_hyperparameters.csv",
    index=False
)

if "XGBoost" in classical_model_parameters:
    print("Full XGBoost settings used by this environment:")
    display(
        pd.DataFrame(
            list(classical_model_parameters["XGBoost"].items()),
            columns=["Parameter", "Value"]
        )
    )

print("Saved classical model hyperparameters.")


In [ ]:
# =========================
# 5.2 Publication-Ready Classical Model Configuration Summary
# =========================
# The baseline settings were fixed before held-out test evaluation.
# No hyperparameter search was performed on the held-out test set.

model_setting_rows = []
for name, model_obj in ml_models.items():
    params = model_obj.get_params(deep=False)
    model_setting_rows.append({
        "Model": name,
        "Key_Settings": json.dumps(params, default=str, sort_keys=True),
        "Tuning_Procedure": "No automated hyperparameter search in the final baseline experiment",
        "Search_Space": "Not applicable",
        "Selection_Metric": "Models compared on common held-out Accuracy, Precision, Recall, F1, and ROC-AUC",
        "Test_Set_Used_For_Tuning": False,
        "Random_State": params.get("random_state", None),
        "Settings_Status": "Fixed before final held-out evaluation",
    })

classical_model_configuration_summary = pd.DataFrame(model_setting_rows)
display(classical_model_configuration_summary)

classical_model_configuration_summary.to_csv(
    RESULTS_DIR / "FINAL_classical_model_configuration_summary.csv",
    index=False
)


In [ ]:
# =========================
# 6. Train and Evaluate All ML Models
# =========================
results = []
trained_models = {}
prediction_outputs = {}
classification_reports = {}

training_times = {}
for name, model in ml_models.items():
    print("\n" + "#" * 90)
    print(f"Training ML model: {name}")
    print("#" * 90)
    start_time = time()

    pipe = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = get_positive_probability(pipe, X_test)

    trained_models[name] = pipe
    prediction_outputs[name] = {
        "y_pred": y_pred,
        "y_proba": y_proba,
        "type": "ML"
    }
    results.append(evaluate_model(name, y_test, y_pred, y_proba))
    classification_reports[name] = save_classification_report(name, y_test, y_pred)
    plot_confusion_matrix(name, y_test, y_pred)
    plot_single_roc(name, y_test, y_proba)

    end_time = time()
    elapsed_time = end_time - start_time

    trained_models[name] = pipe
    training_times[name] = elapsed_time

    print(f"{name} training completed in {elapsed_time:.2f} seconds.")

ml_results_df = pd.DataFrame(results).sort_values("F1_Score", ascending=False)
print("ML model comparison:")
display(ml_results_df.style.format({
    "Accuracy": "{:.6f}",
    "Precision": "{:.6f}",
    "Recall": "{:.6f}",
    "F1_Score": "{:.6f}",
    "ROC_AUC": "{:.6f}",
}))

ml_results_df.to_csv(RESULTS_DIR / "ml_model_comparison.csv", index=False)


## 7. Deep Learning Model

The MLP uses the same baseline train/test split and a separately fitted clone of the preprocessing pipeline. Its architecture, optimizer, callback settings, batch size, validation split, class-weight handling, and classification threshold are defined explicitly below and exported for reproducibility.

The baseline MLP is evaluated once on the common 20% holdout set. Extended cross-validation and multi-seed stability tests are applied to the selected best classical model (XGBoost) to keep the validation workload computationally practical while directly testing the model used for the final decision-support pipeline.


In [ ]:
# =========================
# 7. Train and Evaluate Deep Learning MLP
# =========================
tensorflow_available = False
history = None
try_message = None

MLP_CONFIG = {
    "hidden_units": [128, 64, 32],
    "hidden_activation": "relu",
    "kernel_initializer": "glorot_uniform",
    "bias_initializer": "zeros",
    "batch_normalization_after_layers": [128, 64],
    "dropout_rates": [0.30, 0.25, 0.15],
    "output_units": 1,
    "output_activation": "sigmoid",
    "optimizer": "Adam",
    "initial_learning_rate": 0.001,
    "loss": "binary_crossentropy",
    "reported_training_metrics": ["accuracy", "AUC"],
    "validation_split": 0.20,
    "max_epochs": 100,
    "batch_size": 128,
    "classification_threshold": 0.50,
    "class_weight": "balanced",
    "early_stopping_monitor": "val_auc",
    "early_stopping_mode": "max",
    "early_stopping_patience": 10,
    "restore_best_weights": True,
    "reduce_lr_monitor": "val_auc",
    "reduce_lr_mode": "max",
    "reduce_lr_factor": 0.5,
    "reduce_lr_patience": 4,
    "minimum_learning_rate": 1e-5,
    "random_seed": SEED,
}

with open(RESULTS_DIR / "deep_learning_mlp_hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(MLP_CONFIG, f, indent=2)

print("Deep Learning MLP configuration:")
display(pd.DataFrame(list(MLP_CONFIG.items()), columns=["Parameter", "Value"]))

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers

    tensorflow_available = True
    software_versions["tensorflow"] = tf.__version__

    # Set NumPy/Python/TensorFlow seeds together.
    tf.keras.utils.set_random_seed(SEED)
    try:
        tf.config.experimental.enable_op_determinism()
        software_versions["tensorflow_deterministic_ops"] = True
    except Exception:
        software_versions["tensorflow_deterministic_ops"] = False

    print("TensorFlow available:", tf.__version__)
except Exception as e:
    tensorflow_available = False
    try_message = str(e)
    print("TensorFlow is not available. Skipping deep learning model.")
    print("Reason:", e)

if tensorflow_available:
    start_time = time()

    dl_preprocessor = clone(preprocessor)
    X_train_dl = to_dense(dl_preprocessor.fit_transform(X_train)).astype("float32")
    X_test_dl = to_dense(dl_preprocessor.transform(X_test)).astype("float32")
    dl_feature_names = get_transformed_feature_names(dl_preprocessor)

    classes = np.unique(y_train)
    class_weights_values = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weight_dict = {
        int(cls): float(weight)
        for cls, weight in zip(classes, class_weights_values)
    }
    print("Class weights:", class_weight_dict)

    input_dim = X_train_dl.shape[1]

    dl_model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu", kernel_initializer="glorot_uniform", bias_initializer="zeros"),
        layers.BatchNormalization(),
        layers.Dropout(0.30),
        layers.Dense(64, activation="relu", kernel_initializer="glorot_uniform", bias_initializer="zeros"),
        layers.BatchNormalization(),
        layers.Dropout(0.25),
        layers.Dense(32, activation="relu", kernel_initializer="glorot_uniform", bias_initializer="zeros"),
        layers.Dropout(0.15),
        layers.Dense(1, activation="sigmoid", kernel_initializer="glorot_uniform", bias_initializer="zeros")
    ])

    dl_model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=MLP_CONFIG["initial_learning_rate"]
        ),
        loss=MLP_CONFIG["loss"],
        metrics=["accuracy", keras.metrics.AUC(name="auc")]
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor=MLP_CONFIG["early_stopping_monitor"],
            mode=MLP_CONFIG["early_stopping_mode"],
            patience=MLP_CONFIG["early_stopping_patience"],
            restore_best_weights=MLP_CONFIG["restore_best_weights"]
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor=MLP_CONFIG["reduce_lr_monitor"],
            mode=MLP_CONFIG["reduce_lr_mode"],
            factor=MLP_CONFIG["reduce_lr_factor"],
            patience=MLP_CONFIG["reduce_lr_patience"],
            min_lr=MLP_CONFIG["minimum_learning_rate"]
        )
    ]

    history = dl_model.fit(
        X_train_dl,
        y_train.values,
        validation_split=MLP_CONFIG["validation_split"],
        epochs=MLP_CONFIG["max_epochs"],
        batch_size=MLP_CONFIG["batch_size"],
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )

    dl_proba = dl_model.predict(X_test_dl, verbose=1).ravel()
    dl_pred = (
        dl_proba >= MLP_CONFIG["classification_threshold"]
    ).astype(int)

    model_name = "Deep Learning MLP"

    # IMPORTANT: retain the true Keras model/preprocessor pair.
    # The original notebook accidentally overwrote this dictionary with the
    # last classical ML pipeline, which could create downstream inconsistency.
    trained_models[model_name] = {
        "model": dl_model,
        "preprocessor": dl_preprocessor,
        "feature_names": dl_feature_names,
        "history": history
    }

    prediction_outputs[model_name] = {
        "y_pred": dl_pred,
        "y_proba": dl_proba,
        "type": "DL"
    }

    results.append(
        evaluate_model(model_name, y_test, dl_pred, dl_proba)
    )

    elapsed_time = time() - start_time
    training_times[model_name] = elapsed_time

    print(f"{model_name} training completed in {elapsed_time:.2f} seconds.")
    classification_reports[model_name] = save_classification_report(
        model_name, y_test, dl_pred
    )
    plot_confusion_matrix(model_name, y_test, dl_pred)
    plot_single_roc(model_name, y_test, dl_proba)

    # Training history
    history_df = pd.DataFrame(history.history)
    history_df.to_csv(
        RESULTS_DIR / "deep_learning_training_history.csv",
        index=False
    )

    ax = history_df[["loss", "val_loss"]].plot(figsize=(8, 5))
    ax.set_title("Deep Learning Training and Validation Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Binary Cross-Entropy Loss")
    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / "deep_learning_loss_curve.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    auc_cols = [c for c in history_df.columns if c in ["auc", "val_auc"]]
    if auc_cols:
        ax = history_df[auc_cols].plot(figsize=(8, 5))
        ax.set_title("Deep Learning Training and Validation AUC")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("AUC")
        plt.tight_layout()
        plt.savefig(
            PLOTS_DIR / "deep_learning_auc_curve.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()

    # Save DL assets
    dl_model.save(MODEL_DIR / "deep_learning_delay_model.keras")
    joblib.dump(
        dl_preprocessor,
        MODEL_DIR / "deep_learning_preprocessor.pkl"
    )
    joblib.dump(
        dl_feature_names,
        MODEL_DIR / "deep_learning_feature_names.pkl"
    )

    # Save architecture summary as text.
    summary_lines = []
    dl_model.summary(print_fn=lambda line: summary_lines.append(line))
    (RESULTS_DIR / "deep_learning_model_summary.txt").write_text(
        "\n".join(summary_lines),
        encoding="utf-8"
    )

else:
    print(
        "DL model was not trained. Install TensorFlow to include the MLP "
        "model in comparison and SHAP deep explanation."
    )


In [ ]:
# =========================
# 7.1 MLP Class-Weight Sensitivity
# =========================
# The synthetic target is approximately balanced, so balanced class weights
# should be near 1.0. This cell quantifies whether they materially affect results.

mlp_weight_sensitivity_df = pd.DataFrame()

if tensorflow_available and RUN_MLP_WEIGHT_SENSITIVITY:
    def build_same_mlp(input_dim, seed):
        tf.keras.utils.set_random_seed(seed)
        m = keras.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(128, activation="relu", kernel_initializer="glorot_uniform", bias_initializer="zeros"),
            layers.BatchNormalization(),
            layers.Dropout(0.30),
            layers.Dense(64, activation="relu", kernel_initializer="glorot_uniform", bias_initializer="zeros"),
            layers.BatchNormalization(),
            layers.Dropout(0.25),
            layers.Dense(32, activation="relu", kernel_initializer="glorot_uniform", bias_initializer="zeros"),
            layers.Dropout(0.15),
            layers.Dense(1, activation="sigmoid", kernel_initializer="glorot_uniform", bias_initializer="zeros")
        ])
        m.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss="binary_crossentropy",
            metrics=["accuracy", keras.metrics.AUC(name="auc")]
        )
        return m

    # Baseline result with class weighting already exists in `results`.
    baseline_mlp_row = next(
        (r for r in results if r["Model"] == "Deep Learning MLP"),
        None
    )

    no_weight_model = build_same_mlp(X_train_dl.shape[1], SEED)
    no_weight_callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_auc", mode="max", patience=10, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc", mode="max", factor=0.5, patience=4, min_lr=1e-5
        )
    ]
    no_weight_model.fit(
        X_train_dl,
        y_train.values,
        validation_split=0.20,
        epochs=100,
        batch_size=128,
        callbacks=no_weight_callbacks,
        class_weight=None,
        verbose=1
    )
    no_weight_proba = no_weight_model.predict(X_test_dl, verbose=0).ravel()
    no_weight_pred = (no_weight_proba >= 0.50).astype(int)
    no_weight_row = evaluate_model(
        "MLP_without_class_weight",
        y_test,
        no_weight_pred,
        no_weight_proba
    )

    rows = []
    if baseline_mlp_row is not None:
        weighted_row = dict(baseline_mlp_row)
        weighted_row["Condition"] = "Balanced class weights"
        rows.append(weighted_row)
    no_weight_row["Condition"] = "No class weights"
    rows.append(no_weight_row)

    mlp_weight_sensitivity_df = pd.DataFrame(rows)
    display(mlp_weight_sensitivity_df)

    mlp_weight_sensitivity_df.to_csv(
        RESULTS_DIR / "FINAL_mlp_class_weight_sensitivity.csv",
        index=False
    )
else:
    print("MLP class-weight sensitivity skipped.")


In [ ]:
# =========================
# 8. Compare All Baseline ML and DL Models
# =========================
all_results_df = (
    pd.DataFrame(results)
    .sort_values("F1_Score", ascending=False)
    .reset_index(drop=True)
)

print("Final baseline ML + DL model comparison:")
display(
    all_results_df.style.format({
        "Accuracy": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}",
        "F1_Score": "{:.6f}",
        "ROC_AUC": "{:.6f}",
    })
)

# This file is the single source of truth for the manuscript baseline table.
baseline_results_path = RESULTS_DIR / "FINAL_baseline_model_comparison.csv"
all_results_df.to_csv(baseline_results_path, index=False)
all_results_df.to_csv(
    RESULTS_DIR / "all_ml_dl_model_comparison.csv",
    index=False
)

# Human-readable rounded manuscript table.
manuscript_baseline_df = all_results_df.copy()
for col in ["Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC"]:
    manuscript_baseline_df[col] = manuscript_baseline_df[col].round(4)
manuscript_baseline_df.to_csv(
    RESULTS_DIR / "FINAL_manuscript_baseline_model_table.csv",
    index=False
)

# Bar chart comparison
metric_cols = ["Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC"]
comparison_plot_df = (
    all_results_df
    .set_index("Model")[metric_cols]
    .sort_values("F1_Score", ascending=True)
)
ax = comparison_plot_df.plot(kind="barh", figsize=(11, 7))
ax.set_title("ML and Deep Learning Model Comparison")
ax.set_xlabel("Score")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "all_model_metric_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Combined ROC curve for all baseline models
fig, ax = plt.subplots(figsize=(8, 6))
for name, output in prediction_outputs.items():
    RocCurveDisplay.from_predictions(
        y_test,
        output["y_proba"],
        name=(
            f"{name} "
            f"(AUC={roc_auc_score(y_test, output['y_proba']):.3f})"
        ),
        ax=ax
    )
ax.set_title("Combined ROC Curves - ML and Deep Learning Models")
plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "combined_roc_curves_all_models.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

selected_model_name = all_results_df.iloc[0]["Model"]
best_model_name = selected_model_name  # compatibility alias used by downstream notebooks
best_model_object = trained_models[selected_model_name]

print(
    "Selected downstream model by prespecified F1 ordering:",
    selected_model_name
)
print(
    "Model ordering is descriptive; statistical/practical equivalence is assessed "
    "separately with the paired bootstrap."
)
print("Use manuscript numbers only from:", baseline_results_path)


In [ ]:
# =========================
# 8.0 Paired Uncertainty: XGBoost vs MLP
# =========================
# The observed F1 difference is very small. This paired bootstrap assesses
# uncertainty on the SAME held-out cases instead of declaring a winner from
# a 0.0008 point estimate difference.

paired_bootstrap_df = pd.DataFrame()
paired_bootstrap_summary_df = pd.DataFrame()

if (
    RUN_PAIRED_BOOTSTRAP
    and "XGBoost" in prediction_outputs
    and "Deep Learning MLP" in prediction_outputs
):
    rng_boot = np.random.default_rng(SEED + 900)
    y_true_np = y_test.to_numpy()
    xgb_pred_np = np.asarray(prediction_outputs["XGBoost"]["y_pred"])
    mlp_pred_np = np.asarray(prediction_outputs["Deep Learning MLP"]["y_pred"])
    n_test = len(y_true_np)

    differences = np.empty(BOOTSTRAP_REPS, dtype=float)
    for b in range(BOOTSTRAP_REPS):
        idx_boot = rng_boot.integers(0, n_test, size=n_test)
        f1_xgb = f1_score(
            y_true_np[idx_boot], xgb_pred_np[idx_boot], zero_division=0
        )
        f1_mlp = f1_score(
            y_true_np[idx_boot], mlp_pred_np[idx_boot], zero_division=0
        )
        differences[b] = f1_xgb - f1_mlp

    ci_low, ci_high = np.quantile(differences, [0.025, 0.975])
    observed_diff = (
        f1_score(y_true_np, xgb_pred_np, zero_division=0)
        - f1_score(y_true_np, mlp_pred_np, zero_division=0)
    )

    paired_bootstrap_summary_df = pd.DataFrame([{
        "Metric": "F1_Score",
        "Comparison": "XGBoost - Deep Learning MLP",
        "Observed_Difference": observed_diff,
        "Bootstrap_Repetitions": BOOTSTRAP_REPS,
        "CI_95_Lower": ci_low,
        "CI_95_Upper": ci_high,
        "CI_Includes_Zero": bool(ci_low <= 0 <= ci_high),
        "Recommended_Interpretation": (
            "Practically equivalent on this held-out synthetic benchmark"
            if ci_low <= 0 <= ci_high
            else "Small paired difference detected within this synthetic benchmark"
        )
    }])

    paired_bootstrap_df = pd.DataFrame({
        "Bootstrap_F1_Difference_XGB_minus_MLP": differences
    })
    display(paired_bootstrap_summary_df)

    paired_bootstrap_summary_df.to_csv(
        RESULTS_DIR / "FINAL_xgb_mlp_paired_bootstrap_summary.csv",
        index=False
    )
    paired_bootstrap_df.to_csv(
        RESULTS_DIR / "FINAL_xgb_mlp_paired_bootstrap_distribution.csv",
        index=False
    )
else:
    print("Paired XGBoost-MLP bootstrap skipped.")


## 8.1 Extended Validation of the Selected Model

The baseline 80/20 comparison is retained for comparability across all algorithms. To address the limitations of relying on a single split, the selected XGBoost model is subjected to three additional validation checks:

1. **5-fold stratified cross-validation** using the full synthetic dataset.
2. **Repeated stratified holdout validation** using multiple random seeds.
3. **Synthetic-condition robustness tests** using fresh stochastic target noise, increased target noise, and a predefined adverse distribution shift.

These tests assess stability within the synthetic environment. They do not constitute real-world external validation.


In [ ]:
# =========================
# 8.1 Learning-Curve / Dataset-Size Sensitivity
# =========================
learning_curve_df = pd.DataFrame()

if RUN_LEARNING_CURVE and "XGBoost" in ml_models:
    lc_rows = []

    for total_n in LEARNING_CURVE_TOTAL_SIZES:
        total_n = min(total_n, len(X))

        if total_n < len(X):
            X_subset, _, y_subset, _ = train_test_split(
                X,
                y,
                train_size=total_n,
                random_state=SEED,
                stratify=y
            )
        else:
            X_subset = X.copy()
            y_subset = y.copy()

        X_lc_train, X_lc_test, y_lc_train, y_lc_test = train_test_split(
            X_subset,
            y_subset,
            test_size=0.20,
            random_state=SEED,
            stratify=y_subset
        )

        lc_model = clone(ml_models["XGBoost"])
        lc_model.set_params(random_state=SEED)
        lc_pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", lc_model)
        ])

        t0 = time()
        lc_pipeline.fit(X_lc_train, y_lc_train)
        elapsed = time() - t0

        lc_pred = lc_pipeline.predict(X_lc_test)
        lc_proba = get_positive_probability(lc_pipeline, X_lc_test)
        row = evaluate_model("XGBoost", y_lc_test, lc_pred, lc_proba)
        row.update({
            "Total_Dataset_Size": int(total_n),
            "Training_Rows": int(len(X_lc_train)),
            "Testing_Rows": int(len(X_lc_test)),
            "Training_Seconds": float(elapsed)
        })
        lc_rows.append(row)
        print("Completed learning-curve size:", total_n)

    learning_curve_df = pd.DataFrame(lc_rows).sort_values("Total_Dataset_Size")
    display(learning_curve_df)

    learning_curve_df.to_csv(
        RESULTS_DIR / "FINAL_xgboost_learning_curve.csv",
        index=False
    )

    ax = learning_curve_df.plot(
        x="Total_Dataset_Size",
        y=["F1_Score", "ROC_AUC"],
        marker="o",
        figsize=(8, 5)
    )
    ax.set_xscale("log")
    ax.set_ylim(0, 1)
    ax.set_title("XGBoost Learning Curve Across Synthetic Dataset Sizes")
    ax.set_ylabel("Score")
    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / "FINAL_xgboost_learning_curve.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
else:
    print("Learning-curve analysis skipped.")


In [ ]:
# =========================
# 8.1 Five-Fold Stratified Cross-Validation for XGBoost
# =========================
xgb_cv_fold_df = pd.DataFrame()
xgb_cv_summary_df = pd.DataFrame()

if RUN_EXTENDED_VALIDATION and "XGBoost" in ml_models:
    cv = StratifiedKFold(
        n_splits=CV_SPLITS,
        shuffle=True,
        random_state=SEED
    )

    cv_rows = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        print(f"Running XGBoost CV fold {fold}/{CV_SPLITS}...")

        X_cv_train = X.iloc[train_idx]
        X_cv_valid = X.iloc[valid_idx]
        y_cv_train = y.iloc[train_idx]
        y_cv_valid = y.iloc[valid_idx]

        fold_model = clone(ml_models["XGBoost"])
        fold_model.set_params(random_state=SEED + fold)

        fold_pipeline = Pipeline(steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", fold_model)
        ])

        fold_pipeline.fit(X_cv_train, y_cv_train)
        fold_pred = fold_pipeline.predict(X_cv_valid)
        fold_proba = get_positive_probability(fold_pipeline, X_cv_valid)

        row = evaluate_model(
            f"XGBoost_CV_Fold_{fold}",
            y_cv_valid,
            fold_pred,
            fold_proba
        )
        row["Fold"] = fold
        row["Train_Rows"] = len(train_idx)
        row["Validation_Rows"] = len(valid_idx)
        cv_rows.append(row)

    xgb_cv_fold_df = pd.DataFrame(cv_rows)

    metric_cols_cv = ["Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC"]

    xgb_cv_summary_df = pd.DataFrame({
        "Metric": metric_cols_cv,
        "Mean": [xgb_cv_fold_df[m].mean() for m in metric_cols_cv],
        "Std": [xgb_cv_fold_df[m].std(ddof=1) for m in metric_cols_cv],
        "Min": [xgb_cv_fold_df[m].min() for m in metric_cols_cv],
        "Max": [xgb_cv_fold_df[m].max() for m in metric_cols_cv],
    })

    xgb_cv_fold_df.to_csv(
        RESULTS_DIR / "FINAL_xgboost_5fold_cv_folds.csv",
        index=False
    )
    xgb_cv_summary_df.to_csv(
        RESULTS_DIR / "FINAL_xgboost_5fold_cv_summary.csv",
        index=False
    )

    print("\nXGBoost 5-fold cross-validation results:")
    display(xgb_cv_fold_df)
    print("\nMean ± SD:")
    display(xgb_cv_summary_df)

else:
    print("Extended XGBoost cross-validation skipped.")


In [ ]:
# =========================
# 8.2 Repeated Holdout Validation Across Random Seeds
# =========================
xgb_seed_results_df = pd.DataFrame()
xgb_seed_summary_df = pd.DataFrame()

if RUN_EXTENDED_VALIDATION and "XGBoost" in ml_models:
    seed_rows = []

    for validation_seed in VALIDATION_SEEDS:
        print(f"Running XGBoost repeated holdout, seed={validation_seed}...")

        X_seed_train, X_seed_test, y_seed_train, y_seed_test = train_test_split(
            X,
            y,
            test_size=0.20,
            random_state=validation_seed,
            stratify=y
        )

        seed_model = clone(ml_models["XGBoost"])
        seed_model.set_params(random_state=validation_seed)

        seed_pipeline = Pipeline(steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", seed_model)
        ])

        seed_pipeline.fit(X_seed_train, y_seed_train)
        seed_pred = seed_pipeline.predict(X_seed_test)
        seed_proba = get_positive_probability(seed_pipeline, X_seed_test)

        row = evaluate_model(
            "XGBoost",
            y_seed_test,
            seed_pred,
            seed_proba
        )
        row["Random_Seed"] = validation_seed
        seed_rows.append(row)

    xgb_seed_results_df = pd.DataFrame(seed_rows)

    metric_cols_seed = ["Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC"]

    xgb_seed_summary_df = pd.DataFrame({
        "Metric": metric_cols_seed,
        "Mean": [xgb_seed_results_df[m].mean() for m in metric_cols_seed],
        "Std": [xgb_seed_results_df[m].std(ddof=1) for m in metric_cols_seed],
        "Min": [xgb_seed_results_df[m].min() for m in metric_cols_seed],
        "Max": [xgb_seed_results_df[m].max() for m in metric_cols_seed],
    })

    xgb_seed_results_df.to_csv(
        RESULTS_DIR / "FINAL_xgboost_multiple_seed_holdout.csv",
        index=False
    )
    xgb_seed_summary_df.to_csv(
        RESULTS_DIR / "FINAL_xgboost_multiple_seed_summary.csv",
        index=False
    )

    print("\nXGBoost repeated holdout results:")
    display(xgb_seed_results_df)
    print("\nMean ± SD across random seeds:")
    display(xgb_seed_summary_df)

else:
    print("Multiple-seed validation skipped.")


## 8.3 Robustness Under Changed Synthetic Conditions

The following stress tests preserve the documented target-generation equation from Notebook 01 while changing selected stochastic or feature conditions. The objective is to test whether the baseline XGBoost model remains useful when the synthetic environment is perturbed.

- **Baseline holdout:** original test set and original target.
- **Fresh target noise:** same test features, but the stochastic target term is regenerated with the original noise standard deviation (0.45).
- **Higher target noise:** the stochastic target term is regenerated with a larger standard deviation (0.65).
- **Adverse 0.5-SD distribution shift:** selected delay-related features are shifted by one-half of their training-set standard deviation in the adverse direction; derived approval duration and external-risk score are recalculated before a fresh target is generated.

For each regenerated scenario, the binary threshold is the median scenario probability, preserving a balanced synthetic classification target. These scenarios are sensitivity analyses, not real-world validation.


In [ ]:
# =========================
# 8.3 Synthetic-Condition Robustness Tests
# =========================
def compute_rule_score_for_frame(frame):
    """Reproduce the rule component from Notebook 01."""
    rs = pd.Series(0.0, index=frame.index)

    rs += np.where(
        (frame["Authorities_Involved"] > 4)
        & (frame["Approval_Duration_Days"] > 35),
        0.85, 0
    )
    rs += np.where(
        (frame["Design_Change_Count"] >= 5)
        & (frame["Scope_Clarity_Score"] < 6),
        0.90, 0
    )
    rs += np.where(
        (frame["Stakeholder_Communication_Score"] < 5)
        & (frame["Coordination_Score"] < 5),
        0.80, 0
    )
    rs += np.where(
        (frame["Resource_Availability_Score"] < 5)
        | (frame["Labor_Productivity_Score"] < 5),
        0.65, 0
    )
    rs += np.where(
        (frame["Supplier_Reliability_Score"] < 5)
        & (frame["Procurement_Lead_Time_Days"] > 45),
        0.70, 0
    )
    rs += np.where(
        (frame["Contractor_Financial_Stability"] < 5)
        & (frame["Payment_Delay_Days"] > 20),
        0.75, 0
    )
    rs += np.where(
        (frame["External_Risk_Score"] > 7)
        | (frame["Force_Majeure_Flag"] == 1),
        0.95, 0
    )
    rs += np.where(
        (frame["Schedule_Buffer_Percent"] < 5)
        & (frame["Complexity_Score"] >= 3),
        0.65, 0
    )

    rs -= np.where(
        (frame["BIM_Adoption_Level"] >= 2)
        & (frame["Coordination_Score"] >= 7),
        0.45, 0
    )
    rs -= np.where(
        (frame["AI_Tools_Adoption_Level"] >= 2)
        & (frame["Project_Manager_Experience_Years"] >= 8),
        0.35, 0
    )
    rs -= np.where(
        (frame["Schedule_Buffer_Percent"] >= 12)
        & (frame["Scope_Clarity_Score"] >= 7),
        0.35, 0
    )

    return rs


def regenerate_synthetic_target(frame, seed, noise_sd=0.45):
    """
    Reproduce Notebook 01's continuous score and regenerate only the
    stochastic term and target labels.
    """
    rng_local = np.random.default_rng(seed)
    rs = compute_rule_score_for_frame(frame)

    deterministic_score = (
        -2.40
        + 0.34 * frame["Complexity_Score"]
        + 0.015 * (frame["Planned_Duration_Days"] / 10)
        + 0.020 * np.log1p(frame["Planned_Budget_Million"])
        + 0.10 * frame["Design_Change_Count"]
        + 0.09 * frame["Change_Request_Count"]
        + 0.012 * frame["Approval_Duration_Days"]
        + 0.012 * frame["Procurement_Lead_Time_Days"]
        + 0.018 * frame["Payment_Delay_Days"]
        + 0.11 * frame["External_Risk_Score"]
        + 0.06 * frame["Quality_Defect_Rate"]
        + 0.18 * frame["Safety_Incident_Count"]
        - 0.12 * frame["Scope_Clarity_Score"]
        - 0.10 * frame["Stakeholder_Communication_Score"]
        - 0.09 * frame["Coordination_Score"]
        - 0.08 * frame["Contractor_Experience_Score"]
        - 0.08 * frame["Resource_Availability_Score"]
        - 0.07 * frame["Labor_Productivity_Score"]
        - 0.06 * frame["Equipment_Availability_Score"]
        - 0.06 * frame["Supplier_Reliability_Score"]
        - 0.06 * frame["Contractor_Financial_Stability"]
        - 0.05 * frame["Owner_Decision_Speed_Score"]
        - 0.035 * frame["Schedule_Buffer_Percent"]
        - 0.18 * frame["BIM_Adoption_Level"]
        - 0.12 * frame["AI_Tools_Adoption_Level"]
        + rs
    )

    latent_score = deterministic_score + rng_local.normal(
        0, noise_sd, len(frame)
    )
    probability = 1.0 / (1.0 + np.exp(-latent_score))
    threshold = float(np.median(probability))
    label = (probability >= threshold).astype(int)

    return (
        pd.Series(probability, index=frame.index),
        pd.Series(label, index=frame.index),
        threshold
    )


def apply_adverse_half_sd_shift(frame, reference_frame):
    """
    Apply a transparent 0.5-SD adverse shift to selected variables.
    Bounds follow the synthetic generator's documented ranges.
    """
    shifted = frame.copy()

    increase_specs = {
        "Approval_Per_Authority_Days": (2, 15, True),
        "Procurement_Lead_Time_Days": (5, 120, True),
        "Payment_Delay_Days": (0, 90, True),
        "Weather_Risk_Score": (1, 10, False),
        "Permit_Risk_Score": (1, 10, False),
        "Inflation_Risk_Score": (1, 10, False),
        "Site_Condition_Risk_Score": (1, 10, False),
    }

    decrease_specs = {
        "Scope_Clarity_Score": (1, 10),
        "Stakeholder_Communication_Score": (1, 10),
        "Coordination_Score": (1, 10),
        "Resource_Availability_Score": (1, 10),
        "Labor_Productivity_Score": (1, 10),
        "Supplier_Reliability_Score": (1, 10),
        "Contractor_Financial_Stability": (1, 10),
        "Owner_Decision_Speed_Score": (1, 10),
    }

    for col, (low, high, integer_like) in increase_specs.items():
        delta = 0.5 * reference_frame[col].std(ddof=0)
        shifted[col] = np.clip(shifted[col] + delta, low, high)
        if integer_like:
            shifted[col] = np.rint(shifted[col]).astype(int)
        else:
            shifted[col] = shifted[col].round(1)

    for col, (low, high) in decrease_specs.items():
        delta = 0.5 * reference_frame[col].std(ddof=0)
        shifted[col] = np.clip(shifted[col] - delta, low, high).round(1)

    shifted["Approval_Duration_Days"] = (
        shifted["Authorities_Involved"]
        * shifted["Approval_Per_Authority_Days"]
    )

    shifted["External_Risk_Score"] = (
        0.27 * shifted["Weather_Risk_Score"]
        + 0.30 * shifted["Permit_Risk_Score"]
        + 0.22 * shifted["Inflation_Risk_Score"]
        + 0.21 * shifted["Site_Condition_Risk_Score"]
        + 1.5 * shifted["Force_Majeure_Flag"]
    ).clip(1, 10).round(2)

    return shifted


robustness_results_df = pd.DataFrame()

if RUN_EXTENDED_VALIDATION and "XGBoost" in trained_models:
    baseline_xgb = trained_models["XGBoost"]
    robustness_rows = []

    original_pred = prediction_outputs["XGBoost"]["y_pred"]
    original_proba = prediction_outputs["XGBoost"]["y_proba"]
    row = evaluate_model(
        "XGBoost",
        y_test,
        original_pred,
        original_proba
    )
    row.update({
        "Scenario": "Original holdout",
        "Target_Noise_SD": 0.45,
        "Scenario_Threshold": np.nan,
        "Positive_Class_Rate": float(y_test.mean()),
        "Evaluation_Type": "Original target"
    })
    robustness_rows.append(row)

    _, fresh_y, fresh_threshold = regenerate_synthetic_target(
        X_test.copy(),
        seed=142,
        noise_sd=0.45
    )
    fresh_pred = baseline_xgb.predict(X_test)
    fresh_model_proba = get_positive_probability(
        baseline_xgb, X_test
    )
    row = evaluate_model(
        "XGBoost",
        fresh_y,
        fresh_pred,
        fresh_model_proba
    )
    row.update({
        "Scenario": "Fresh target noise",
        "Target_Noise_SD": 0.45,
        "Scenario_Threshold": fresh_threshold,
        "Positive_Class_Rate": float(fresh_y.mean()),
        "Evaluation_Type": "Regenerated synthetic target"
    })
    robustness_rows.append(row)

    _, noisy_y, noisy_threshold = regenerate_synthetic_target(
        X_test.copy(),
        seed=143,
        noise_sd=0.65
    )
    noisy_pred = baseline_xgb.predict(X_test)
    noisy_model_proba = get_positive_probability(
        baseline_xgb, X_test
    )
    row = evaluate_model(
        "XGBoost",
        noisy_y,
        noisy_pred,
        noisy_model_proba
    )
    row.update({
        "Scenario": "Higher target noise",
        "Target_Noise_SD": 0.65,
        "Scenario_Threshold": noisy_threshold,
        "Positive_Class_Rate": float(noisy_y.mean()),
        "Evaluation_Type": "Regenerated synthetic target"
    })
    robustness_rows.append(row)

    X_adverse = apply_adverse_half_sd_shift(
        X_test.copy(),
        X_train
    )
    _, adverse_y, adverse_threshold = regenerate_synthetic_target(
        X_adverse,
        seed=144,
        noise_sd=0.45
    )
    adverse_pred = baseline_xgb.predict(X_adverse)
    adverse_model_proba = get_positive_probability(
        baseline_xgb,
        X_adverse
    )
    row = evaluate_model(
        "XGBoost",
        adverse_y,
        adverse_pred,
        adverse_model_proba
    )
    row.update({
        "Scenario": "Adverse 0.5-SD distribution shift",
        "Target_Noise_SD": 0.45,
        "Scenario_Threshold": adverse_threshold,
        "Positive_Class_Rate": float(adverse_y.mean()),
        "Evaluation_Type": "Shifted features + regenerated target"
    })
    robustness_rows.append(row)

    robustness_results_df = pd.DataFrame(robustness_rows)
    robustness_results_df.to_csv(
        RESULTS_DIR / "FINAL_xgboost_synthetic_robustness.csv",
        index=False
    )

    print("Synthetic-condition robustness results:")
    display(robustness_results_df)

else:
    print("Synthetic-condition robustness tests skipped.")


In [ ]:
# =========================
# 8.4 Structural Generator and Rule-Ablation Sensitivity
# =========================
# This goes beyond fresh noise/feature shift by changing the target-generating
# mechanism itself. It tests transfer of the baseline model and performance
# after retraining under the changed mechanism.

def compute_rule_score_variant(frame, mode="baseline"):
    rs = pd.Series(0.0, index=frame.index)

    if mode == "no_rules":
        return rs

    scale = 0.5 if mode == "half_rules" else 1.0

    # Shared baseline-style rules.
    rs += scale * np.where(
        (frame["Authorities_Involved"] > 4)
        & (frame["Approval_Duration_Days"] > 35), 0.85, 0
    )
    rs += scale * np.where(
        (frame["Design_Change_Count"] >= 5)
        & (frame["Scope_Clarity_Score"] < 6), 0.90, 0
    )
    rs += scale * np.where(
        (frame["Stakeholder_Communication_Score"] < 5)
        & (frame["Coordination_Score"] < 5), 0.80, 0
    )
    rs += scale * np.where(
        (frame["Resource_Availability_Score"] < 5)
        | (frame["Labor_Productivity_Score"] < 5), 0.65, 0
    )
    rs += scale * np.where(
        (frame["Supplier_Reliability_Score"] < 5)
        & (frame["Procurement_Lead_Time_Days"] > 45), 0.70, 0
    )

    if mode == "alternative_structure":
        rs += np.where(
            (frame["Contractor_Financial_Stability"] < 5)
            & (frame["Payment_Delay_Days"] > 30), 0.50, 0
        )
    else:
        rs += scale * np.where(
            (frame["Contractor_Financial_Stability"] < 5)
            & (frame["Payment_Delay_Days"] > 20), 0.75, 0
        )

    rs += scale * np.where(
        (frame["External_Risk_Score"] > 7)
        | (frame["Force_Majeure_Flag"] == 1), 0.95, 0
    )
    rs += scale * np.where(
        (frame["Schedule_Buffer_Percent"] < 5)
        & (frame["Complexity_Score"] >= 3), 0.65, 0
    )

    # Protective rules. Two are intentionally removed in alternative_structure.
    if mode != "alternative_structure":
        rs -= scale * np.where(
            (frame["BIM_Adoption_Level"] >= 2)
            & (frame["Coordination_Score"] >= 7), 0.45, 0
        )

    rs -= scale * np.where(
        (frame["AI_Tools_Adoption_Level"] >= 2)
        & (frame["Project_Manager_Experience_Years"] >= 8), 0.35, 0
    )

    if mode != "alternative_structure":
        rs -= scale * np.where(
            (frame["Schedule_Buffer_Percent"] >= 12)
            & (frame["Scope_Clarity_Score"] >= 7), 0.35, 0
        )

    return pd.Series(rs, index=frame.index)


def generate_target_structural_variant(
    frame,
    seed,
    mode="baseline",
    prevalence=0.50,
    noise_sd=0.45
):
    rng_local = np.random.default_rng(seed)

    coeff = {
        "complexity": 0.34,
        "approval": 0.012,
        "procurement": 0.012,
        "payment": 0.018,
        "coordination": -0.09,
        "bim": -0.18,
    }

    if mode == "alternative_structure":
        coeff.update({
            "complexity": 0.22,
            "approval": 0.007,
            "procurement": 0.020,
            "payment": 0.027,
            "coordination": -0.13,
            "bim": -0.08,
        })

    rs = compute_rule_score_variant(frame, mode=mode)

    deterministic_score = (
        -2.40
        + coeff["complexity"] * frame["Complexity_Score"]
        + 0.015 * (frame["Planned_Duration_Days"] / 10)
        + 0.020 * np.log1p(frame["Planned_Budget_Million"])
        + 0.10 * frame["Design_Change_Count"]
        + 0.09 * frame["Change_Request_Count"]
        + coeff["approval"] * frame["Approval_Duration_Days"]
        + coeff["procurement"] * frame["Procurement_Lead_Time_Days"]
        + coeff["payment"] * frame["Payment_Delay_Days"]
        + 0.11 * frame["External_Risk_Score"]
        + 0.06 * frame["Quality_Defect_Rate"]
        + 0.18 * frame["Safety_Incident_Count"]
        - 0.12 * frame["Scope_Clarity_Score"]
        - 0.10 * frame["Stakeholder_Communication_Score"]
        + coeff["coordination"] * frame["Coordination_Score"]
        - 0.08 * frame["Contractor_Experience_Score"]
        - 0.08 * frame["Resource_Availability_Score"]
        - 0.07 * frame["Labor_Productivity_Score"]
        - 0.06 * frame["Equipment_Availability_Score"]
        - 0.06 * frame["Supplier_Reliability_Score"]
        - 0.06 * frame["Contractor_Financial_Stability"]
        - 0.05 * frame["Owner_Decision_Speed_Score"]
        - 0.035 * frame["Schedule_Buffer_Percent"]
        + coeff["bim"] * frame["BIM_Adoption_Level"]
        - 0.12 * frame["AI_Tools_Adoption_Level"]
        + rs
    )

    latent = deterministic_score + rng_local.normal(0, noise_sd, len(frame))
    probability = 1.0 / (1.0 + np.exp(-latent))

    threshold = float(np.quantile(probability, 1.0 - prevalence))
    label = (probability >= threshold).astype(int)

    return (
        pd.Series(probability, index=frame.index),
        pd.Series(label, index=frame.index),
        threshold
    )


structural_robustness_df = pd.DataFrame()

if RUN_STRUCTURAL_ROBUSTNESS and "XGBoost" in trained_models:
    rows = []
    baseline_xgb = trained_models["XGBoost"]

    for mode in ["no_rules", "half_rules", "alternative_structure"]:
        _, y_variant_all, threshold_variant = generate_target_structural_variant(
            X,
            seed=SEED + 700,
            mode=mode,
            prevalence=0.50,
            noise_sd=0.45
        )

        y_variant_train = y_variant_all.loc[X_train.index]
        y_variant_test = y_variant_all.loc[X_test.index]

        # Transfer: baseline model trained on original mechanism.
        transfer_pred = baseline_xgb.predict(X_test)
        transfer_proba = get_positive_probability(baseline_xgb, X_test)
        transfer_row = evaluate_model(
            "Baseline_XGBoost_transfer",
            y_variant_test,
            transfer_pred,
            transfer_proba
        )
        transfer_row.update({
            "Structural_Variant": mode,
            "Evaluation_Mode": "Transfer without retraining",
            "Target_Threshold": threshold_variant,
            "Positive_Class_Rate": float(y_variant_test.mean())
        })
        rows.append(transfer_row)

        # Retrain XGBoost under the changed mechanism.
        variant_model = clone(ml_models["XGBoost"])
        variant_model.set_params(random_state=SEED + 701)
        variant_pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", variant_model)
        ])
        variant_pipeline.fit(X_train, y_variant_train)
        variant_pred = variant_pipeline.predict(X_test)
        variant_proba = get_positive_probability(variant_pipeline, X_test)

        retrain_row = evaluate_model(
            "XGBoost_retrained_on_variant",
            y_variant_test,
            variant_pred,
            variant_proba
        )
        retrain_row.update({
            "Structural_Variant": mode,
            "Evaluation_Mode": "Retrained within changed mechanism",
            "Target_Threshold": threshold_variant,
            "Positive_Class_Rate": float(y_variant_test.mean())
        })
        rows.append(retrain_row)

    structural_robustness_df = pd.DataFrame(rows)
    display(structural_robustness_df)

    structural_robustness_df.to_csv(
        RESULTS_DIR / "FINAL_xgboost_structural_generator_sensitivity.csv",
        index=False
    )
else:
    print("Structural-generator sensitivity skipped.")


In [ ]:
# =========================
# 8.5 Artificial-Prevalence Sensitivity
# =========================
# The baseline 50/50 target is a designed benchmark. This experiment changes
# prevalence by moving the threshold on the same synthetic probability surface.
# It does NOT estimate real-world prevalence.

prevalence_results_df = pd.DataFrame()
prevalence_explanation_stability_df = pd.DataFrame()

if RUN_PREVALENCE_SENSITIVITY and "XGBoost" in ml_models:
    prevalence_rows = []
    explanation_rows = []

    # Baseline permutation ranking for comparison.
    baseline_perm_sample = X_test.sample(
        n=min(3000, len(X_test)),
        random_state=SEED
    )
    baseline_perm_y = y_test.loc[baseline_perm_sample.index]
    baseline_perm_obj = permutation_importance(
        trained_models["XGBoost"],
        baseline_perm_sample,
        baseline_perm_y,
        n_repeats=5,
        random_state=SEED,
        scoring="f1",
        n_jobs=-1
    )
    baseline_top10 = set(
        pd.DataFrame({
            "Feature": features,
            "Importance": baseline_perm_obj.importances_mean
        })
        .sort_values("Importance", ascending=False)
        .head(10)["Feature"]
    )

    for prevalence in PREVALENCE_LEVELS:
        threshold = float(
            df["Delay_Probability_True"].quantile(1.0 - prevalence)
        )
        y_prev = (
            df["Delay_Probability_True"] >= threshold
        ).astype(int)

        X_prev_train, X_prev_test, y_prev_train, y_prev_test = train_test_split(
            X,
            y_prev,
            test_size=0.20,
            random_state=SEED,
            stratify=y_prev
        )

        prev_model = clone(ml_models["XGBoost"])
        prev_model.set_params(random_state=SEED)
        prev_pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", prev_model)
        ])
        prev_pipeline.fit(X_prev_train, y_prev_train)

        prev_pred = prev_pipeline.predict(X_prev_test)
        prev_proba = get_positive_probability(prev_pipeline, X_prev_test)
        row = evaluate_model(
            "XGBoost",
            y_prev_test,
            prev_pred,
            prev_proba
        )
        row.update({
            "Designed_Delayed_Prevalence": prevalence,
            "Observed_Test_Prevalence": float(y_prev_test.mean()),
            "Synthetic_Probability_Threshold": threshold
        })
        prevalence_rows.append(row)

        perm_sample = X_prev_test.sample(
            n=min(3000, len(X_prev_test)),
            random_state=SEED
        )
        perm_y = y_prev_test.loc[perm_sample.index]
        perm_obj = permutation_importance(
            prev_pipeline,
            perm_sample,
            perm_y,
            n_repeats=5,
            random_state=SEED,
            scoring="f1",
            n_jobs=-1
        )
        perm_rank = (
            pd.DataFrame({
                "Feature": features,
                "Importance": perm_obj.importances_mean
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )
        top10 = set(perm_rank.head(10)["Feature"])
        overlap = len(top10.intersection(baseline_top10))

        explanation_rows.append({
            "Designed_Delayed_Prevalence": prevalence,
            "Top10_Permutation_Overlap_With_50pct_Baseline": overlap,
            "Top10_Overlap_Percent": 100 * overlap / 10,
            "Top10_Features": "; ".join(perm_rank.head(10)["Feature"].tolist())
        })

    prevalence_results_df = pd.DataFrame(prevalence_rows)
    prevalence_explanation_stability_df = pd.DataFrame(explanation_rows)

    display(prevalence_results_df)
    display(prevalence_explanation_stability_df)

    prevalence_results_df.to_csv(
        RESULTS_DIR / "FINAL_xgboost_prevalence_sensitivity.csv",
        index=False
    )
    prevalence_explanation_stability_df.to_csv(
        RESULTS_DIR / "FINAL_xgboost_prevalence_explanation_stability.csv",
        index=False
    )
else:
    print("Prevalence sensitivity skipped.")


In [ ]:
# =========================
# 8.6 Probability Calibration and Distribution-Shift Calibration
# =========================
calibration_summary_df = pd.DataFrame()
calibration_curve_df = pd.DataFrame()

if RUN_CALIBRATION_ANALYSIS and "XGBoost" in ml_models:
    def safe_logit(p):
        p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p)).reshape(-1, 1)

    # Clean three-way split: model-fit subset, calibration subset, held-out test.
    X_fit, X_cal, y_fit, y_cal = train_test_split(
        X_train,
        y_train,
        test_size=0.10,
        random_state=SEED + 810,
        stratify=y_train
    )

    calibration_base_model = clone(ml_models["XGBoost"])
    calibration_base_model.set_params(random_state=SEED + 811)
    calibration_base_pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", calibration_base_model)
    ])
    calibration_base_pipeline.fit(X_fit, y_fit)

    p_cal_raw = get_positive_probability(calibration_base_pipeline, X_cal)

    # Platt/logistic recalibration on log-odds of probabilities.
    probability_calibrator = LogisticRegression(
        solver="lbfgs",
        random_state=SEED + 812
    )
    probability_calibrator.fit(safe_logit(p_cal_raw), y_cal)

    def apply_probability_calibrator(raw_probability):
        return probability_calibrator.predict_proba(
            safe_logit(raw_probability)
        )[:, 1]

    p_test_raw = get_positive_probability(
        calibration_base_pipeline,
        X_test
    )
    p_test_calibrated = apply_probability_calibrator(p_test_raw)

    calibration_rows = [{
        "Evaluation_Set": "Original holdout",
        "Probability_Type": "Raw",
        "Brier_Score": brier_score_loss(y_test, p_test_raw),
        "ROC_AUC": roc_auc_score(y_test, p_test_raw)
    }, {
        "Evaluation_Set": "Original holdout",
        "Probability_Type": "Platt-calibrated",
        "Brier_Score": brier_score_loss(y_test, p_test_calibrated),
        "ROC_AUC": roc_auc_score(y_test, p_test_calibrated)
    }]

    curve_rows = []
    for label, p_values in [
        ("Original holdout - Raw", p_test_raw),
        ("Original holdout - Platt-calibrated", p_test_calibrated)
    ]:
        prob_true, prob_pred = calibration_curve(
            y_test,
            p_values,
            n_bins=10,
            strategy="quantile"
        )
        for bin_id, (pp, pt) in enumerate(zip(prob_pred, prob_true), start=1):
            curve_rows.append({
                "Curve": label,
                "Bin": bin_id,
                "Mean_Predicted_Probability": pp,
                "Observed_Fraction_Positive": pt
            })

    # Calibration under the previously defined adverse feature shift.
    if "X_adverse" in globals() and "adverse_y" in globals():
        p_shift_raw = get_positive_probability(
            calibration_base_pipeline,
            X_adverse
        )
        p_shift_calibrated = apply_probability_calibrator(p_shift_raw)

        calibration_rows.extend([
            {
                "Evaluation_Set": "Adverse 0.5-SD distribution shift",
                "Probability_Type": "Raw",
                "Brier_Score": brier_score_loss(adverse_y, p_shift_raw),
                "ROC_AUC": roc_auc_score(adverse_y, p_shift_raw)
            },
            {
                "Evaluation_Set": "Adverse 0.5-SD distribution shift",
                "Probability_Type": "Baseline Platt-calibrated",
                "Brier_Score": brier_score_loss(adverse_y, p_shift_calibrated),
                "ROC_AUC": roc_auc_score(adverse_y, p_shift_calibrated)
            }
        ])

        for label, y_curve, p_values in [
            ("Adverse shift - Raw", adverse_y, p_shift_raw),
            ("Adverse shift - Baseline Platt-calibrated", adverse_y, p_shift_calibrated)
        ]:
            prob_true, prob_pred = calibration_curve(
                y_curve,
                p_values,
                n_bins=10,
                strategy="quantile"
            )
            for bin_id, (pp, pt) in enumerate(zip(prob_pred, prob_true), start=1):
                curve_rows.append({
                    "Curve": label,
                    "Bin": bin_id,
                    "Mean_Predicted_Probability": pp,
                    "Observed_Fraction_Positive": pt
                })

    calibration_summary_df = pd.DataFrame(calibration_rows)
    calibration_curve_df = pd.DataFrame(curve_rows)

    display(calibration_summary_df)
    display(calibration_curve_df)

    calibration_summary_df.to_csv(
        RESULTS_DIR / "FINAL_probability_calibration_summary.csv",
        index=False
    )
    calibration_curve_df.to_csv(
        RESULTS_DIR / "FINAL_probability_calibration_curve_points.csv",
        index=False
    )

    fig, ax = plt.subplots(figsize=(7, 6))
    for curve_name, group in calibration_curve_df.groupby("Curve"):
        ax.plot(
            group["Mean_Predicted_Probability"],
            group["Observed_Fraction_Positive"],
            marker="o",
            label=curve_name
        )
    ax.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed fraction positive")
    ax.set_title("Probability Calibration")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / "FINAL_probability_calibration_curves.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    # Save clean calibration artifacts for Notebook 03.
    joblib.dump(
        calibration_base_pipeline,
        MODEL_DIR / "scenario_probability_base_pipeline.pkl"
    )
    joblib.dump(
        probability_calibrator,
        MODEL_DIR / "scenario_probability_platt_calibrator.pkl"
    )

    calibration_metadata = {
        "calibration_method": "Platt/logistic scaling on logit(raw_probability)",
        "model_fit_rows": int(len(X_fit)),
        "calibration_rows": int(len(X_cal)),
        "held_out_test_rows": int(len(X_test)),
        "calibration_seed": SEED + 810,
        "interpretation": (
            "Calibration is internal to the baseline synthetic mechanism. "
            "Shift results assess sensitivity and do not establish real-world calibration."
        )
    }
    with open(
        RESULTS_DIR / "FINAL_probability_calibration_metadata.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(calibration_metadata, f, indent=2)
else:
    print("Probability-calibration analysis skipped.")


In [ ]:
# =========================
# 8.4 Consolidated Validation Summary
# =========================
validation_summary = {
    "baseline_best_model": best_model_name,
    "baseline_seed": SEED,
    "baseline_test_fraction": 0.20,
    "cv_splits": CV_SPLITS,
    "multiple_seed_values": VALIDATION_SEEDS,
    "extended_validation_run": RUN_EXTENDED_VALIDATION,
    "interpretation_scope": (
        "Validation is internal to the synthetic environment and does not "
        "constitute real-world external validation."
    )
}

if not xgb_cv_summary_df.empty:
    validation_summary["xgboost_5fold_cv"] = {
        row["Metric"]: {
            "mean": float(row["Mean"]),
            "std": float(row["Std"]),
            "min": float(row["Min"]),
            "max": float(row["Max"]),
        }
        for _, row in xgb_cv_summary_df.iterrows()
    }

if not xgb_seed_summary_df.empty:
    validation_summary["xgboost_multiple_seed_holdout"] = {
        row["Metric"]: {
            "mean": float(row["Mean"]),
            "std": float(row["Std"]),
            "min": float(row["Min"]),
            "max": float(row["Max"]),
        }
        for _, row in xgb_seed_summary_df.iterrows()
    }

with open(
    RESULTS_DIR / "FINAL_validation_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(validation_summary, f, indent=2)

print("Validation summary saved to:")
print(RESULTS_DIR / "FINAL_validation_summary.json")


## 9. Permutation Importance for Best Classical ML Model

This section uses permutation importance on the best classical ML pipeline. If the best overall model is the DL model, the notebook still selects the best available classical ML model for this feature-importance output.


In [ ]:
# =========================
# 9. Permutation Feature Importance for Best Classical ML Model
# =========================
ml_only_results = all_results_df[all_results_df["Model"].isin(ml_models.keys())].copy()
if len(ml_only_results) > 0:
    best_ml_model_name = ml_only_results.iloc[0]["Model"]
    best_ml_pipeline = trained_models[best_ml_model_name]
    print("Best classical ML model:", best_ml_model_name)

    sample_size = min(2000, len(X_test))
    X_imp = X_test.sample(sample_size, random_state=SEED)
    y_imp = y_test.loc[X_imp.index]

    perm = permutation_importance(
        best_ml_pipeline,
        X_imp,
        y_imp,
        n_repeats=8,
        random_state=SEED,
        scoring="f1",
        n_jobs=-1
    )

    importance_df = pd.DataFrame({
        "Feature": features,
        "Importance_Mean": perm.importances_mean,
        "Importance_Std": perm.importances_std
    }).sort_values("Importance_Mean", ascending=False)

    importance_df["Importance_Percentage"] = (
        importance_df["Importance_Mean"].clip(lower=0) /
        importance_df["Importance_Mean"].clip(lower=0).sum() * 100
    )

    display(importance_df.head(20))

    ax = importance_df.head(15).sort_values("Importance_Mean").plot(
        x="Feature", y="Importance_Mean", kind="barh", figsize=(9, 6), legend=False
    )
    ax.set_title(f"Permutation Feature Importance - {best_ml_model_name}")
    ax.set_xlabel("Mean decrease in F1 when shuffled")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "permutation_importance_best_ml_model.png", dpi=300, bbox_inches="tight")
    plt.show()

    importance_df.to_csv(RESULTS_DIR / "permutation_feature_importance_best_ml_model.csv", index=False)
else:
    print("No classical ML model was available for permutation importance.")


## 10. SHAP Explainability

This section applies SHAP on three model families:

1. **Linear SHAP** for Logistic Regression.
2. **Tree SHAP** for Random Forest or another available tree-based model.
3. **Deep SHAP / Gradient SHAP fallback** for the Keras deep learning model when TensorFlow and SHAP are compatible.

The notebook uses samples to keep runtime practical. Increase the sample sizes if your machine can handle longer execution time.


In [ ]:
# =========================
# 10. SHAP Utilities
# =========================
def aggregate_shap_to_main_features(shap_values_array, transformed_feature_names):
    """Aggregate one-hot encoded SHAP values back to original main feature names."""
    shap_abs_mean = np.abs(shap_values_array).mean(axis=0)
    rows = []
    for feature_name, value in zip(transformed_feature_names, shap_abs_mean):
        main_feature = feature_name
        for cat in categorical_features:
            if feature_name.startswith(cat + "_"):
                main_feature = cat
                break
        rows.append({"Main_Feature": main_feature, "Mean_ABS_SHAP": float(value)})

    return (
        pd.DataFrame(rows)
        .groupby("Main_Feature", as_index=False)["Mean_ABS_SHAP"]
        .sum()
        .sort_values("Mean_ABS_SHAP", ascending=False)
    )


def normalize_binary_shap_values(shap_values):
    """Return a 2D SHAP matrix for the positive class."""
    if isinstance(shap_values, list):
        return shap_values[1] if len(shap_values) > 1 else shap_values[0]
    arr = np.asarray(shap_values)
    # Some explainers return shape: (samples, features, classes)
    if arr.ndim == 3:
        return arr[:, :, 1] if arr.shape[-1] > 1 else arr[:, :, 0]
    return arr


def plot_shap_bar(grouped_df, title, filename):
    ax = grouped_df.head(15).sort_values("Mean_ABS_SHAP").plot(
        x="Main_Feature", y="Mean_ABS_SHAP", kind="barh", figsize=(9, 6), legend=False
    )
    ax.set_title(title)
    ax.set_xlabel("Mean |SHAP value|")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# =========================
# 10.1 SHAP for One Linear Model: Logistic Regression
# =========================
try:
    import shap

    shap_sample_raw = safe_sample_frame(X_test, n=500)
    linear_pipeline = trained_models.get("Logistic Regression")

    if linear_pipeline is None:
        print("Logistic Regression was not trained, so linear SHAP is skipped.")
    else:
        linear_pre = linear_pipeline.named_steps["preprocessor"]
        linear_model = linear_pipeline.named_steps["model"]

        X_train_linear_bg = safe_sample_frame(X_train, n=200)
        X_bg_linear = to_dense(linear_pre.transform(X_train_linear_bg))
        X_shap_linear = to_dense(linear_pre.transform(shap_sample_raw))
        linear_feature_names = get_transformed_feature_names(linear_pre)

        linear_explainer = shap.LinearExplainer(linear_model, X_bg_linear)
        linear_shap_values = normalize_binary_shap_values(linear_explainer.shap_values(X_shap_linear))

        linear_shap_feature_df = pd.DataFrame({
            "Feature": linear_feature_names,
            "Mean_ABS_SHAP": np.abs(linear_shap_values).mean(axis=0)
        }).sort_values("Mean_ABS_SHAP", ascending=False)
        linear_shap_grouped_df = aggregate_shap_to_main_features(linear_shap_values, linear_feature_names)

        print("Top Linear SHAP features - Logistic Regression")
        display(linear_shap_feature_df.head(20))
        display(linear_shap_grouped_df.head(20))

        shap.summary_plot(
            linear_shap_values,
            X_shap_linear,
            feature_names=linear_feature_names,
            show=False,
            max_display=20
        )
        plt.title("SHAP Summary Plot - Logistic Regression")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / "shap_summary_linear_logistic_regression.png", dpi=300, bbox_inches="tight")
        plt.show()

        plot_shap_bar(
            linear_shap_grouped_df,
            "Grouped SHAP Importance - Logistic Regression",
            "shap_grouped_bar_linear_logistic_regression.png"
        )

        linear_shap_feature_df.to_csv(RESULTS_DIR / "shap_linear_logistic_transformed_features.csv", index=False)
        linear_shap_grouped_df.to_csv(RESULTS_DIR / "shap_linear_logistic_grouped_features.csv", index=False)
except Exception as e:
    print("Linear SHAP section skipped. Reason:", e)


In [ ]:
# =========================
# 10.2 SHAP for XGBoost Only
# =========================

try:
    import shap

    xgb_model_name = "XGBoost"

    if xgb_model_name not in trained_models:
        print("XGBoost model was not trained, so XGBoost SHAP is skipped.")
    else:
        xgb_pipeline = trained_models[xgb_model_name]

        # Extract preprocessing and model from pipeline
        xgb_pre = xgb_pipeline.named_steps["preprocessor"]
        xgb_model = xgb_pipeline.named_steps["model"]

        # Sample test data for faster SHAP execution
        shap_sample_raw = safe_sample_frame(X_test, n=500)

        # Transform test sample using the same preprocessing pipeline
        X_shap_xgb = to_dense(xgb_pre.transform(shap_sample_raw))

        # Get transformed feature names after OneHotEncoding / scaling
        xgb_feature_names = get_transformed_feature_names(xgb_pre)

        # Create SHAP TreeExplainer specifically for XGBoost
        xgb_explainer = shap.TreeExplainer(xgb_model)

        # Compute SHAP values
        xgb_shap_values_raw = xgb_explainer.shap_values(X_shap_xgb)

        # Normalize SHAP values for binary or multiclass output
        xgb_shap_values = normalize_binary_shap_values(xgb_shap_values_raw)

        # Create transformed-feature SHAP importance table
        xgb_shap_feature_df = pd.DataFrame({
            "Feature": xgb_feature_names,
            "Mean_ABS_SHAP": np.abs(xgb_shap_values).mean(axis=0)
        }).sort_values("Mean_ABS_SHAP", ascending=False)

        # Aggregate one-hot encoded/transformed features back to main feature names
        xgb_shap_grouped_df = aggregate_shap_to_main_features(
            xgb_shap_values,
            xgb_feature_names
        )

        print("Top XGBoost SHAP features - transformed features")
        display(xgb_shap_feature_df.head(20))

        print("Top XGBoost SHAP features - grouped main features")
        display(xgb_shap_grouped_df.head(20))

        # SHAP summary dot plot
        shap.summary_plot(
            xgb_shap_values,
            X_shap_xgb,
            feature_names=xgb_feature_names,
            show=False,
            max_display=20
        )

        plt.title("SHAP Summary Plot - XGBoost")
        plt.tight_layout()
        plt.savefig(
            PLOTS_DIR / "shap_summary_tree_xgboost.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()

        # Grouped SHAP bar plot
        plot_shap_bar(
            xgb_shap_grouped_df,
            "Grouped SHAP Importance - XGBoost",
            "shap_grouped_bar_tree_xgboost.png"
        )

        # Save SHAP outputs
        xgb_shap_feature_df.to_csv(
            RESULTS_DIR / "shap_tree_xgboost_transformed_features.csv",
            index=False
        )

        xgb_shap_grouped_df.to_csv(
            RESULTS_DIR / "shap_tree_xgboost_grouped_features.csv",
            index=False
        )

        print("XGBoost SHAP analysis completed successfully.")

except Exception as e:
    print("XGBoost SHAP section skipped. Reason:", e)


In [ ]:
# =========================
# 10.3 SHAP for Deep Learning MLP
# Robust version: forces explainer to use a real TensorFlow/Keras DL model
# =========================

try:
    import shap
    import numpy as np
    import pandas as pd
    import tensorflow as tf
    from sklearn.pipeline import Pipeline

    # --------------------------------------------------
    # Helper: safely resolve the REAL deep learning model
    # --------------------------------------------------
    def resolve_deep_learning_assets():
        dl_model_local = None
        dl_preprocessor_local = None
        dl_source = None

        # 1) First try trained_models["Deep Learning MLP"]
        if "Deep Learning MLP" in trained_models:
            candidate = trained_models["Deep Learning MLP"]

            if isinstance(candidate, dict):
                cand_model = candidate.get("model", None)
                cand_pre = candidate.get("preprocessor", None)

                if isinstance(cand_model, tf.keras.Model):
                    dl_model_local = cand_model
                    dl_preprocessor_local = cand_pre
                    dl_source = "trained_models['Deep Learning MLP'] as dict"

            elif isinstance(candidate, Pipeline):
                cand_model = candidate.named_steps.get("model", None)
                cand_pre = candidate.named_steps.get("preprocessor", None)

                if isinstance(cand_model, tf.keras.Model):
                    dl_model_local = cand_model
                    dl_preprocessor_local = cand_pre
                    dl_source = "trained_models['Deep Learning MLP'] as Pipeline"

        # 2) If not found, try global variables commonly used for DL models
        if dl_model_local is None:
            candidate_model_names = [
                "dl_model",
                "deep_learning_model",
                "mlp_model",
                "keras_model",
                "deep_model"
            ]

            for var_name in candidate_model_names:
                if var_name in globals():
                    cand_model = globals()[var_name]
                    if isinstance(cand_model, tf.keras.Model):
                        dl_model_local = cand_model
                        dl_source = f"global variable '{var_name}'"
                        break

        # 3) Resolve preprocessor if not already found
        if dl_preprocessor_local is None:
            candidate_preprocessor_names = [
                "dl_preprocessor",
                "preprocessor"
            ]

            for var_name in candidate_preprocessor_names:
                if var_name in globals():
                    dl_preprocessor_local = globals()[var_name]
                    break

        return dl_model_local, dl_preprocessor_local, dl_source

    # --------------------------------------------------
    # Main checks
    # --------------------------------------------------
    if not tensorflow_available:
        print("TensorFlow is not available, so Deep Learning SHAP is skipped.")

    else:
        dl_model, dl_preprocessor, dl_source = resolve_deep_learning_assets()

        if dl_model is None:
            raise ValueError(
                "No real TensorFlow/Keras deep learning model was found. "
                "Your trained_models['Deep Learning MLP'] appears to be overwritten or stored incorrectly."
            )

        if dl_preprocessor is None:
            raise ValueError(
                "No valid preprocessor was found for the deep learning model."
            )

        print(f"Deep Learning model source: {dl_source}")
        print("Resolved DL model type:", type(dl_model))
        print("Resolved preprocessor type:", type(dl_preprocessor))

        if not isinstance(dl_model, tf.keras.Model):
            raise TypeError(
                f"Resolved model is not a TensorFlow/Keras model. Got: {type(dl_model)}"
            )

        # --------------------------------------------------
        # Regenerate feature names from the same preprocessor
        # --------------------------------------------------
        dl_feature_names = get_transformed_feature_names(dl_preprocessor)

        # --------------------------------------------------
        # Prepare SHAP data
        # --------------------------------------------------
        X_bg_raw = safe_sample_frame(X_train, n=100)
        X_explain_raw = safe_sample_frame(X_test, n=300)

        X_bg_deep = to_dense(
            dl_preprocessor.transform(X_bg_raw)
        ).astype("float32")

        X_explain_deep = to_dense(
            dl_preprocessor.transform(X_explain_raw)
        ).astype("float32")

        if X_explain_deep.shape[1] != len(dl_feature_names):
            raise ValueError(
                f"Feature-name mismatch before SHAP: transformed data has "
                f"{X_explain_deep.shape[1]} columns, but feature names has "
                f"{len(dl_feature_names)} names."
            )

        # --------------------------------------------------
        # Run SHAP explainers
        # --------------------------------------------------
        deep_shap_values = None
        deep_method_used = None

        try:
            deep_explainer = shap.DeepExplainer(dl_model, X_bg_deep)
            deep_shap_values = deep_explainer.shap_values(X_explain_deep)
            deep_method_used = "DeepExplainer"

        except Exception as deep_error:
            print("DeepExplainer failed; trying GradientExplainer fallback.")
            print("DeepExplainer reason:", deep_error)

            try:
                deep_explainer = shap.GradientExplainer(dl_model, X_bg_deep)
                deep_shap_values = deep_explainer.shap_values(X_explain_deep)
                deep_method_used = "GradientExplainer"

            except Exception as grad_error:
                print("GradientExplainer failed; trying KernelExplainer fallback on a smaller sample.")
                print("GradientExplainer reason:", grad_error)

                X_bg_kernel = X_bg_deep[:50]
                X_explain_kernel = X_explain_deep[:80]

                def kernel_predict(data):
                    return dl_model.predict(data, verbose=0).ravel()

                deep_explainer = shap.KernelExplainer(
                    kernel_predict,
                    X_bg_kernel
                )

                deep_shap_values = deep_explainer.shap_values(
                    X_explain_kernel,
                    nsamples=100
                )

                X_explain_deep = X_explain_kernel
                deep_method_used = "KernelExplainer"

        # --------------------------------------------------
        # Normalize SHAP output shape
        # --------------------------------------------------
        deep_shap_values = np.array(deep_shap_values)

        print("Raw deep_shap_values shape:", deep_shap_values.shape)
        print("X_explain_deep shape:", X_explain_deep.shape)
        print("Number of feature names:", len(dl_feature_names))

        # Case A: (1, samples, features) or (2, samples, features)
        if deep_shap_values.ndim == 3 and deep_shap_values.shape[0] in [1, 2]:
            deep_shap_values = deep_shap_values[-1]

        # Case B: (samples, features, 1)
        elif deep_shap_values.ndim == 3 and deep_shap_values.shape[-1] == 1:
            deep_shap_values = deep_shap_values[:, :, 0]

        # Case C: (samples, features, classes)
        elif deep_shap_values.ndim == 3 and deep_shap_values.shape[-1] in [1, 2]:
            deep_shap_values = deep_shap_values[:, :, -1]

        # Case D: (classes, samples, features, 1)
        elif deep_shap_values.ndim == 4 and deep_shap_values.shape[-1] == 1:
            deep_shap_values = deep_shap_values[..., 0]
            if deep_shap_values.shape[0] in [1, 2]:
                deep_shap_values = deep_shap_values[-1]

        print("Final deep_shap_values shape:", deep_shap_values.shape)
        print("Final X_explain_deep shape:", X_explain_deep.shape)

        if deep_shap_values.ndim != 2:
            raise ValueError(
                f"Expected 2D SHAP values after normalization, but got shape "
                f"{deep_shap_values.shape}."
            )

        if deep_shap_values.shape[1] != len(dl_feature_names):
            raise ValueError(
                f"Feature-name mismatch after SHAP: SHAP has "
                f"{deep_shap_values.shape[1]} features, but feature names has "
                f"{len(dl_feature_names)} names."
            )

        if X_explain_deep.shape[1] != len(dl_feature_names):
            raise ValueError(
                f"Feature-name mismatch for summary plot: X_explain_deep has "
                f"{X_explain_deep.shape[1]} features, but feature names has "
                f"{len(dl_feature_names)} names."
            )

        # --------------------------------------------------
        # SHAP importance tables
        # --------------------------------------------------
        deep_shap_feature_df = pd.DataFrame({
            "Feature": dl_feature_names,
            "Mean_ABS_SHAP": np.abs(deep_shap_values).mean(axis=0)
        }).sort_values("Mean_ABS_SHAP", ascending=False)

        deep_shap_grouped_df = aggregate_shap_to_main_features(
            deep_shap_values,
            dl_feature_names
        )

        print(f"Top Deep Learning SHAP features - method used: {deep_method_used}")
        display(deep_shap_feature_df.head(20))

        print("Top Deep Learning SHAP grouped main features")
        display(deep_shap_grouped_df.head(20))

        # --------------------------------------------------
        # SHAP summary plot
        # --------------------------------------------------
        shap.summary_plot(
            deep_shap_values,
            X_explain_deep,
            feature_names=dl_feature_names,
            show=False,
            max_display=20
        )

        plt.title(f"SHAP Summary Plot - Deep Learning MLP ({deep_method_used})")
        plt.tight_layout()
        plt.savefig(
            PLOTS_DIR / "shap_summary_deep_learning_mlp.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()

        # --------------------------------------------------
        # Grouped SHAP bar plot
        # --------------------------------------------------
        plot_shap_bar(
            deep_shap_grouped_df,
            f"Grouped SHAP Importance - Deep Learning MLP ({deep_method_used})",
            "shap_grouped_bar_deep_learning_mlp.png"
        )

        # --------------------------------------------------
        # Save SHAP results
        # --------------------------------------------------
        deep_shap_feature_df.to_csv(
            RESULTS_DIR / "shap_deep_learning_transformed_features.csv",
            index=False
        )

        deep_shap_grouped_df.to_csv(
            RESULTS_DIR / "shap_deep_learning_grouped_features.csv",
            index=False
        )

        print("Deep Learning SHAP analysis completed successfully.")

except Exception as e:
    print("Deep SHAP section skipped. Reason:", e)

## 11. Local Sensitivity and Local SHAP Explanation for One High-Risk Synthetic Project

This section performs model-based local sensitivity analysis and SHAP explanation for one high-risk synthetic test case. The perturbations are interpreted as **scenario-based predicted probability changes**, not causal treatment effects.


In [ ]:
# =========================
# 11. Local Sensitivity + SHAP Explanation for One High-Risk Project
# =========================

def predict_delay_probability_any_model(model_name, project_frame):
    """Predict delay probability using either a sklearn pipeline or the DL model."""
    model_obj = trained_models[model_name]

    if isinstance(model_obj, dict):
        matrix = to_dense(
            model_obj["preprocessor"].transform(project_frame[features])
        ).astype("float32")

        return float(
            model_obj["model"].predict(matrix, verbose=0).ravel()[0]
        )

    return float(model_obj.predict_proba(project_frame[features])[0, 1])


def local_sensitivity_explanation(project_row, model_name, candidate_features=None):
    """Estimate local feature influence by perturbing one feature at a time.
    Negative probability change means the tested change reduces delay risk.
    """
    if candidate_features is None:
        candidate_features = [
            "Scope_Clarity_Score",
            "Stakeholder_Communication_Score",
            "Coordination_Score",
            "Contractor_Experience_Score",
            "Resource_Availability_Score",
            "Labor_Productivity_Score",
            "Equipment_Availability_Score",
            "Supplier_Reliability_Score",
            "Contractor_Financial_Stability",
            "Owner_Decision_Speed_Score",
            "Change_Request_Count",
            "Authorities_Involved",
            "Approval_Per_Authority_Days",
            "Approval_Duration_Days",
            "BIM_Adoption_Level",
            "AI_Tools_Adoption_Level",
            "Schedule_Buffer_Percent",
            "Planned_Budget_Million",
            "Planned_Duration_Days"
        ]

    original = project_row.iloc[[0]].copy()
    base_probability = predict_delay_probability_any_model(model_name, original)
    rows = []

    for col in candidate_features:
        if col not in original.columns:
            continue

        adjusted = original.copy()
        current = adjusted.iloc[0][col]

        if col == "Change_Request_Count":
            adjusted[col] = max(0, int(round(current - 1)))

        elif col == "Authorities_Involved":
            adjusted[col] = max(1, int(round(current - 1)))
            adjusted["Approval_Duration_Days"] = (
                adjusted["Authorities_Involved"]
                * adjusted["Approval_Per_Authority_Days"]
            )

        elif col == "Approval_Per_Authority_Days":
            adjusted[col] = max(2, int(round(current - 1)))
            adjusted["Approval_Duration_Days"] = (
                adjusted["Authorities_Involved"]
                * adjusted["Approval_Per_Authority_Days"]
            )

        elif col == "Approval_Duration_Days":
            # Derived field: do not perturb directly because it must remain
            # consistent with Authorities_Involved × Approval_Per_Authority_Days.
            continue

        elif col in [
            "Planned_Budget_Million",
            "Planned_Duration_Days"
        ]:
            # For this scenario-based decision-support logic, only test an increase, capped at 10%.
            adjusted[col] = current * 1.10

        elif col in [
            "BIM_Adoption_Level",
            "AI_Tools_Adoption_Level"
        ]:
            adjusted[col] = min(3, current + 1)

        else:
            adjusted[col] = min(10, current + 1)

        new_probability = predict_delay_probability_any_model(
            model_name,
            adjusted
        )

        rows.append({
            "Model": model_name,
            "Feature": col,
            "Original_Value": current,
            "Tested_Value": adjusted.iloc[0][col],
            "Base_Probability": base_probability,
            "New_Probability": new_probability,
            "Probability_Change": new_probability - base_probability,
            "Risk_Reduction": base_probability - new_probability
        })

    return pd.DataFrame(rows).sort_values(
        "Risk_Reduction",
        ascending=False
    )


# =========================
# Select One High-Risk Project
# =========================

best_probs = prediction_outputs[best_model_name]["y_proba"]
selected_position = int(np.argmax(best_probs))
selected_index = X_test.index[selected_position]
selected_project = X_test.loc[[selected_index]].copy()

local_xai_df = local_sensitivity_explanation(
    selected_project,
    best_model_name
)

display(selected_project)
display(local_xai_df.head(15))


# =========================
# Local Sensitivity Plot
# =========================

ax = local_xai_df.head(12).sort_values("Risk_Reduction").plot(
    x="Feature",
    y="Risk_Reduction",
    kind="barh",
    figsize=(9, 6),
    legend=False
)

ax.set_title(f"Local Sensitivity XAI - {best_model_name}")
ax.set_xlabel("Delay Probability Reduction")

plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "local_sensitivity_high_risk_project.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

local_xai_df.to_csv(
    RESULTS_DIR / "xai_local_sensitivity_high_risk_project.csv",
    index=False
)


# =========================
# 11.2 SHAP Waterfall and Force Plot for Same High-Risk Project
# =========================

try:
    import shap
    import numpy as np
    import pandas as pd

    print(f"\nRunning local SHAP explanation for: {best_model_name}")

    selected_project_raw = selected_project[features].copy()

    model_obj = trained_models[best_model_name]

    # --------------------------------------------------
    # Case 1: Deep Learning MLP stored as dictionary
    # --------------------------------------------------
    if isinstance(model_obj, dict):
        shap_preprocessor = model_obj["preprocessor"]
        shap_model = model_obj["model"]

        shap_feature_names = model_obj.get(
            "feature_names",
            get_transformed_feature_names(shap_preprocessor)
        )

        X_train_sample_raw = safe_sample_frame(X_train[features], n=100)
        X_background = to_dense(
            shap_preprocessor.transform(X_train_sample_raw)
        ).astype("float32")

        X_selected_shap = to_dense(
            shap_preprocessor.transform(selected_project_raw)
        ).astype("float32")

        def shap_predict_fn(data):
            return shap_model.predict(data, verbose=0).ravel()

        # KernelExplainer is slower but reliable for DL local explanation.
        local_explainer = shap.KernelExplainer(
            shap_predict_fn,
            X_background[:50]
        )

        shap_values_raw = local_explainer.shap_values(
            X_selected_shap,
            nsamples=100
        )

        expected_value = local_explainer.expected_value

        shap_values_local = np.array(shap_values_raw)

        if isinstance(expected_value, (list, np.ndarray)):
            expected_value = np.array(expected_value).ravel()[0]

    # --------------------------------------------------
    # Case 2: sklearn pipeline model
    # --------------------------------------------------
    else:
        shap_preprocessor = model_obj.named_steps["preprocessor"]
        shap_estimator = model_obj.named_steps["model"]

        shap_feature_names = get_transformed_feature_names(shap_preprocessor)

        X_train_sample_raw = safe_sample_frame(X_train[features], n=300)
        X_background = to_dense(
            shap_preprocessor.transform(X_train_sample_raw)
        )

        X_selected_shap = to_dense(
            shap_preprocessor.transform(selected_project_raw)
        )

        # Tree models: Random Forest, Extra Trees, XGBoost, Decision Tree, etc.
        tree_model_keywords = [
            "Random Forest",
            "Extra Trees",
            "XGBoost",
            "Gradient Boosting",
            "Hist Gradient Boosting",
            "Decision Tree"
        ]

        if any(keyword in best_model_name for keyword in tree_model_keywords):
            local_explainer = shap.TreeExplainer(shap_estimator)
            shap_values_raw = local_explainer.shap_values(X_selected_shap)
            expected_value = local_explainer.expected_value

        # Linear model: Logistic Regression
        elif "Logistic Regression" in best_model_name:
            local_explainer = shap.LinearExplainer(
                shap_estimator,
                X_background
            )
            shap_values_raw = local_explainer.shap_values(X_selected_shap)
            expected_value = local_explainer.expected_value

        # Fallback for any other sklearn model
        else:
            def shap_predict_fn(data):
                return shap_estimator.predict_proba(data)[:, 1]

            local_explainer = shap.KernelExplainer(
                shap_predict_fn,
                X_background[:50]
            )

            shap_values_raw = local_explainer.shap_values(
                X_selected_shap,
                nsamples=100
            )

            expected_value = local_explainer.expected_value

        shap_values_local = np.array(shap_values_raw)

    # =========================
    # Normalize SHAP Values
    # =========================

    print("Raw local SHAP shape:", shap_values_local.shape)
    print("Selected transformed shape:", X_selected_shap.shape)
    print("Feature names:", len(shap_feature_names))

    # Case: list-like binary output converted to array:
    # (2, samples, features) or (1, samples, features)
    if shap_values_local.ndim == 3 and shap_values_local.shape[0] in [1, 2]:
        shap_values_local = shap_values_local[-1]

    # Case: (samples, features, 1)
    elif shap_values_local.ndim == 3 and shap_values_local.shape[-1] == 1:
        shap_values_local = shap_values_local[:, :, 0]

    # Case: (samples, features, classes)
    elif shap_values_local.ndim == 3 and shap_values_local.shape[-1] in [1, 2]:
        shap_values_local = shap_values_local[:, :, -1]

    # Final local single row
    if shap_values_local.ndim == 2:
        shap_values_single = shap_values_local[0]
    elif shap_values_local.ndim == 1:
        shap_values_single = shap_values_local
    else:
        raise ValueError(
            f"Unsupported local SHAP shape after normalization: {shap_values_local.shape}"
        )

    X_selected_single = X_selected_shap[0]

    # Normalize expected value
    expected_value = np.array(expected_value)

    if expected_value.ndim > 0:
        if expected_value.size == 1:
            expected_value_single = float(expected_value.ravel()[0])
        else:
            # For binary classification, use positive class expected value
            expected_value_single = float(expected_value.ravel()[-1])
    else:
        expected_value_single = float(expected_value)

    # Validate feature count
    if len(shap_values_single) != len(shap_feature_names):
        raise ValueError(
            f"SHAP feature mismatch: SHAP has {len(shap_values_single)} values, "
            f"but feature names has {len(shap_feature_names)} names."
        )

    if len(X_selected_single) != len(shap_feature_names):
        raise ValueError(
            f"Selected row mismatch: selected transformed row has "
            f"{len(X_selected_single)} values, but feature names has "
            f"{len(shap_feature_names)} names."
        )

    # =========================
    # Build SHAP Explanation Object
    # =========================

    local_explanation = shap.Explanation(
        values=shap_values_single,
        base_values=expected_value_single,
        data=X_selected_single,
        feature_names=shap_feature_names
    )

    # =========================
    # SHAP Waterfall Plot
    # =========================

    shap.plots.waterfall(
        local_explanation,
        max_display=20,
        show=False
    )

    plt.title(f"SHAP Waterfall Plot - {best_model_name}")
    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / "shap_waterfall_high_risk_project.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    # =========================
    # SHAP Force Plot - HTML
    # =========================

    shap.initjs()

    force_plot = shap.force_plot(
        expected_value_single,
        shap_values_single,
        X_selected_single,
        feature_names=shap_feature_names,
        matplotlib=False
    )

    shap.save_html(
        str(PLOTS_DIR / "shap_force_high_risk_project.html"),
        force_plot
    )

    print(
        "SHAP force plot saved to:",
        PLOTS_DIR / "shap_force_high_risk_project.html"
    )

    # =========================
    # Optional: Static Force Plot PNG
    # =========================
    # This may work better for reports/PDFs, but it can be visually crowded.

    shap.force_plot(
        expected_value_single,
        shap_values_single,
        X_selected_single,
        feature_names=shap_feature_names,
        matplotlib=True,
        show=False
    )

    plt.title(f"SHAP Force Plot - {best_model_name}")
    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / "shap_force_high_risk_project.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    # =========================
    # Save Local SHAP Values Table
    # =========================

    local_shap_df = pd.DataFrame({
        "Feature": shap_feature_names,
        "Feature_Value": X_selected_single,
        "SHAP_Value": shap_values_single,
        "ABS_SHAP_Value": np.abs(shap_values_single)
    }).sort_values("ABS_SHAP_Value", ascending=False)

    display(local_shap_df.head(20))

    local_shap_df.to_csv(
        RESULTS_DIR / "xai_local_shap_high_risk_project.csv",
        index=False
    )

    print("Local SHAP waterfall and force plots completed successfully.")

except Exception as e:
    print("Local SHAP waterfall/force section skipped. Reason:", e)

In [ ]:
# =========================
# 11.5 Reproducible Environment and Hardware Specification
# =========================
hardware_metadata = {
    "platform": platform.platform(),
    "system": platform.system(),
    "release": platform.release(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "cpu_logical_count": os.cpu_count(),
}

try:
    import psutil
    hardware_metadata["ram_gb"] = round(
        psutil.virtual_memory().total / (1024 ** 3), 2
    )
    software_versions["psutil"] = psutil.__version__
except Exception:
    hardware_metadata["ram_gb"] = None

if tensorflow_available:
    try:
        hardware_metadata["tensorflow_visible_gpus"] = [
            d.name for d in tf.config.list_physical_devices("GPU")
        ]
    except Exception:
        hardware_metadata["tensorflow_visible_gpus"] = []
else:
    hardware_metadata["tensorflow_visible_gpus"] = []

# Compact requirements file containing the main reproducibility dependencies.
requirements_lines = [
    f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"joblib=={joblib.__version__}",
]
if "xgboost" in software_versions:
    requirements_lines.append(f"xgboost=={software_versions['xgboost']}")
if "tensorflow" in software_versions:
    requirements_lines.append(f"tensorflow=={software_versions['tensorflow']}")
try:
    import shap as _shap_env
    requirements_lines.append(f"shap=={_shap_env.__version__}")
except Exception:
    pass
requirements_lines.append("matplotlib")

(RESULTS_DIR / "requirements.txt").write_text(
    "\n".join(requirements_lines) + "\n",
    encoding="utf-8"
)

environment_yml = [
    "name: project-delay-publication",
    "channels:",
    "  - conda-forge",
    "dependencies:",
    f"  - python={sys.version_info.major}.{sys.version_info.minor}",
    "  - pip",
    "  - pip:",
]
environment_yml.extend([f"      - {line}" for line in requirements_lines])

(RESULTS_DIR / "environment.yml").write_text(
    "\n".join(environment_yml) + "\n",
    encoding="utf-8"
)

with open(
    RESULTS_DIR / "FINAL_hardware_environment.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(hardware_metadata, f, indent=2)

print("Environment files written:")
print("-", RESULTS_DIR / "requirements.txt")
print("-", RESULTS_DIR / "environment.yml")
print("-", RESULTS_DIR / "FINAL_hardware_environment.json")
display(pd.Series(hardware_metadata, name="Value"))


In [ ]:
# =========================
# 11.6 Public Repository Release Manifest
# =========================
# A permanent GitHub/Zenodo URL and DOI cannot be created by the notebook.
# This cell prepares the exact artifact list to deposit before journal submission.

repository_manifest = pd.DataFrame([
    ["notebooks/01_generate_synthetic_dataset.ipynb", "Synthetic data generator and audit tables"],
    ["notebooks/02_train_validate_models.ipynb", "Model comparison, validation, calibration, robustness"],
    ["notebooks/03_scenario_decision_support.ipynb", "Calibrated scenario sensitivity analysis"],
    ["notebooks/04_explainable_ai.ipynb", "XAI and synthetic-ground-truth comparison"],
    ["synthetic_project_delay_dataset.csv", "Synthetic benchmark dataset"],
    ["data_dictionary.csv", "Data dictionary"],
    ["FINAL_generation_specification_with_rationale.csv", "Generator specification and rationale"],
    ["FINAL_target_coefficient_rationale.csv", "Target coefficient audit"],
    ["FINAL_target_rule_rationale.csv", "Interaction-rule audit"],
    ["requirements.txt", "Pinned Python dependencies"],
    ["environment.yml", "Reproducible conda environment"],
    ["FINAL_hardware_environment.json", "Execution environment metadata"],
], columns=["Artifact", "Purpose"])

repository_manifest["Suggested_License"] = "CC BY 4.0 for data/docs; MIT or BSD-3-Clause for code (authors to choose)"
repository_manifest["Repository_Status"] = "Deposit before submission; replace placeholder with permanent URL/DOI"

repository_manifest.to_csv(
    RESULTS_DIR / "FINAL_public_repository_manifest.csv",
    index=False
)

readme_template = """# Project Delay Risk Prediction — Reproducibility Package

This repository contains the synthetic dataset generator, model-training and
validation code, probability-calibration analysis, explainability analysis,
scenario-sensitivity engine, environment specification, and supporting outputs
for the associated manuscript.

## Important scope statement
The dataset is synthetic. Numerical generator settings are simulation
assumptions unless explicitly supported by a cited source. Reported model
performance is internal to the synthetic benchmark and is not external
validation on Saudi/Vision 2030 or other organizational projects.

## Run order
1. notebooks/01_generate_synthetic_dataset.ipynb
2. notebooks/02_train_validate_models.ipynb
3. notebooks/03_scenario_decision_support.ipynb
4. notebooks/04_explainable_ai.ipynb

## Permanent archive
Zenodo DOI: [ADD AFTER DEPOSIT]
Repository URL: [ADD AFTER DEPOSIT]
License: [AUTHORS TO SELECT]
"""
(RESULTS_DIR / "README_repository_template.md").write_text(
    readme_template,
    encoding="utf-8"
)

print(
    "Repository package manifest prepared. "
    "A public URL/DOI must still be created externally before submission."
)
display(repository_manifest)


In [ ]:
# =========================
# 12. Save Best Model and Reproducibility Metadata
# =========================
# Save best sklearn pipeline if the best model is ML.
# If the best model is DL, save its Keras model and preprocessor separately.
if isinstance(best_model_object, dict):
    best_model_object["model"].save(
        MODEL_DIR / "best_delay_prediction_deep_learning_model.keras"
    )
    joblib.dump(
        best_model_object["preprocessor"],
        MODEL_DIR / "best_delay_prediction_deep_learning_preprocessor.pkl"
    )
    best_artifact_path = (
        MODEL_DIR / "best_delay_prediction_deep_learning_model.keras"
    )
else:
    joblib.dump(
        best_model_object,
        MODEL_DIR / "best_delay_prediction_pipeline.pkl"
    )
    best_artifact_path = MODEL_DIR / "best_delay_prediction_pipeline.pkl"

metadata = {
    "best_model_name": best_model_name,
    "features": features,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "target": target,
    "classification_threshold": 0.50,
    "baseline_seed": SEED,
    "baseline_test_fraction": 0.20,
    "cv_splits": CV_SPLITS,
    "validation_seeds": VALIDATION_SEEDS,
    "all_trained_models": list(trained_models.keys()),
    "baseline_results_file": str(
        RESULTS_DIR / "FINAL_baseline_model_comparison.csv"
    ),
    "manuscript_baseline_table": str(
        RESULTS_DIR / "FINAL_manuscript_baseline_model_table.csv"
    ),
    "cv_summary_file": str(
        RESULTS_DIR / "FINAL_xgboost_5fold_cv_summary.csv"
    ),
    "multiple_seed_summary_file": str(
        RESULTS_DIR / "FINAL_xgboost_multiple_seed_summary.csv"
    ),
    "robustness_results_file": str(
        RESULTS_DIR / "FINAL_xgboost_synthetic_robustness.csv"
    ),
    "classical_hyperparameters_file": str(
        RESULTS_DIR / "classical_model_hyperparameters.json"
    ),
    "mlp_hyperparameters_file": str(
        RESULTS_DIR / "deep_learning_mlp_hyperparameters.json"
    ),
    "software_versions": software_versions,
    "interpretation_scope": (
        "The reported validation assesses performance within a synthetic "
        "proof-of-concept environment and requires real-world validation."
    ),
}

joblib.dump(metadata, MODEL_DIR / "model_metadata.pkl")

with open(
    RESULTS_DIR / "FINAL_model_reproducibility_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2, default=str)

# Save all sklearn model pipelines separately for later notebooks.
for name, model_obj in trained_models.items():
    safe_name = name.lower().replace(" ", "_").replace("/", "_")
    if not isinstance(model_obj, dict):
        joblib.dump(
            model_obj,
            MODEL_DIR / f"{safe_name}_pipeline.pkl"
        )

print("Saved best model artifact to:", best_artifact_path)
print("Saved metadata to:", MODEL_DIR / "model_metadata.pkl")
print(
    "Saved baseline model comparison to:",
    RESULTS_DIR / "FINAL_baseline_model_comparison.csv"
)
print("Saved extended validation outputs to:", RESULTS_DIR)
print("Saved plots to:", PLOTS_DIR)
